## Boschertown racing analysis

September 27, 2025

Includes Apple Watch data, using the Sensor Logger app, which can export all sensors in indivisual CSV files, a combined file, sqlite db, etc.

db_session_1 = './Boschertown_Go-Karts-2025-09-27_00-42-19.sqlite'
db_session_2 = './Boschertown_Go-Karts-2025-09-27_00-51-18.sqlite'

In [ ]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
db_session_1 = './Boschertown_Go-Karts-2025-09-27_00-42-19.sqlite'
conn = sqlite3.connect(db_session_1)

# Get the list of tables in the database
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(tables_query, conn)

# Display all tables in the database
print("Tables in the database:")
print(tables)

# For each table, list its columns
for table_name in tables['name']:
    print(f"\nColumns in table '{table_name}':")
    table_info_query = f"PRAGMA table_info({table_name});"
    table_info = pd.read_sql_query(table_info_query, conn)
    print(table_info[['name', 'type']])  # Display column name and type

conn.close()

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import savgol_filter
import folium
from IPython.display import display, HTML

# Set better styling
plt.style.use('dark_background')
sns.set_style("darkgrid")

# Connect to the database
db_session_1 = './Boschertown_Go-Karts-2025-09-27_00-42-19.sqlite'
conn = sqlite3.connect(db_session_1)

# Get metadata
metadata = pd.read_sql_query("SELECT * FROM Metadata", conn)
print("Session Info:")
print(f"Date: {metadata['recording_time'][0]}")
print(f"Device: {metadata['device_name'][0]}")
print(f"Platform: {metadata['platform'][0]}")
print(f"App Version: {metadata['appVersion'][0]}")

# Load key tables
location_df = pd.read_sql_query("SELECT * FROM Location", conn)
accel_df = pd.read_sql_query("SELECT * FROM Accelerometer", conn)
gyro_df = pd.read_sql_query("SELECT * FROM Gyroscope", conn)
speed_df = location_df[['time', 'seconds_elapsed', 'speed', 'latitude', 'longitude', 'altitude']]

# Instead of directly converting to datetime objects, let's first examine the timestamps
print("Timestamp ranges:")
print(f"Location time range: {location_df['time'].min()} to {location_df['time'].max()}")
print(f"Accelerometer time range: {accel_df['time'].min()} to {accel_df['time'].max()}")
print(f"Gyroscope time range: {gyro_df['time'].min()} to {gyro_df['time'].max()}")

# For analysis, let's use seconds_elapsed instead of datetime conversion
# This avoids the timestamp conversion issue while still allowing time-based analysis

# Normalize the time to start from 0 for better plotting
start_time = min(location_df['time'].min(), 
                accel_df['time'].min(), 
                gyro_df['time'].min())

location_df['normalized_time'] = (location_df['time'] - start_time) / 1000  # convert to seconds
accel_df['normalized_time'] = (accel_df['time'] - start_time) / 1000
gyro_df['normalized_time'] = (gyro_df['time'] - start_time) / 1000

# Use normalized_time or seconds_elapsed for plotting instead of datetime
# Replace all instances of 'datetime' with 'seconds_elapsed' or 'normalized_time' in the code

# For example, instead of:
# ax1.plot(speed_df['datetime'], speed_df['smooth_speed'], 'cyan', linewidth=2)
# Use:
# ax1.plot(speed_df['seconds_elapsed'], speed_df['smooth_speed'], 'cyan', linewidth=2)

# Calculate G-forces (1G = 9.81 m/s²)
accel_df['g_force'] = np.sqrt(accel_df['x']**2 + accel_df['y']**2 + accel_df['z']**2) / 9.81
accel_df['lateral_g'] = accel_df['y'] / 9.81
accel_df['longitudinal_g'] = accel_df['x'] / 9.81

# Convert speed to km/h from m/s
speed_df['speed_kmh'] = speed_df['speed'] * 3.6

# Smooth the speed data
speed_df['smooth_speed'] = savgol_filter(speed_df['speed_kmh'], 
                                        window_length=15, 
                                        polyorder=3)

# 1. Overview Dashboard
plt.figure(figsize=(20, 10))
plt.suptitle('Go-Kart Session Overview', fontsize=22, y=0.98)

# Speed over time
ax1 = plt.subplot(2, 2, 1)
ax1.plot(speed_df['seconds_elapsed'], speed_df['smooth_speed'], 'cyan', linewidth=2)
ax1.set_title('Speed vs Time', fontsize=14)
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Speed (km/h)')
ax1.grid(True, alpha=0.3)

# G-Force over time
ax2 = plt.subplot(2, 2, 2)
ax2.plot(accel_df['seconds_elapsed'], accel_df['g_force'], 'lime', linewidth=1.5)
ax2.set_title('G-Force vs Time', fontsize=14)
ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('G-Force (g)')
ax2.grid(True, alpha=0.3)

# Lateral vs Longitudinal G-Force (G-G Plot)
ax3 = plt.subplot(2, 2, 3)
scatter = ax3.scatter(accel_df['lateral_g'], accel_df['longitudinal_g'], 
                     c=accel_df['g_force'], s=10, alpha=0.6, cmap='viridis')
ax3.set_title('G-G Plot (Cornering vs Acceleration/Braking)', fontsize=14)
ax3.set_xlabel('Lateral G-Force (g)')
ax3.set_ylabel('Longitudinal G-Force (g)')
ax3.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax3.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax3.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax3, label='Total G-Force')

# Altitude profile
ax4 = plt.subplot(2, 2, 4)
ax4.plot(speed_df['seconds_elapsed'], speed_df['altitude'], 'magenta', linewidth=1.5)
ax4.set_title('Altitude Profile', fontsize=14)
ax4.set_xlabel('Time (seconds)')
ax4.set_ylabel('Altitude (m)')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

# 2. Track Map with Speed Coloring
if location_df['latitude'].notna().any() and location_df['longitude'].notna().any():
    # Option 1: Use matplotlib for track visualization (this should always work)
    plt.figure(figsize=(12, 10))
    plt.scatter(location_df['longitude'], location_df['latitude'], 
               c=location_df['speed']*3.6, cmap='turbo', 
               s=5, alpha=0.6)
    plt.colorbar(label='Speed (km/h)')
    plt.title('Track Map with Speed', fontsize=16)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(alpha=0.3)
    plt.axis('equal')
    plt.show()
    
    # Option 3: Use plotly with open-street-map which doesn't require a token
    try:
        fig = px.scatter_mapbox(location_df, 
                               lat='latitude', 
                               lon='longitude',
                               color='speed',
                               color_continuous_scale='turbo',
                               hover_data=['speed', 'seconds_elapsed'],
                               zoom=15,
                               title='Track Map with Speed')
        
        # Use open-street-map instead of mapbox styles (no token needed)
        fig.update_layout(
            mapbox_style="open-street-map",
            height=700
        )
        
        fig.update_layout(margin=dict(l=0, r=0, t=30, b=0))
        fig.show()
    except Exception as e:
        print(f"Interactive map could not be displayed: {e}")
        print("Using static map visualization only.")

# 3. Lap Analysis - Attempt to identify laps based on position
# This is an approximation since we don't have explicit lap markers

# For this example, we'll use a simple algorithm based on when the driver crosses
# a specific point on the track (if it's a circuit)
if location_df['latitude'].notna().any() and location_df['longitude'].notna().any():
    # Estimate the start/finish line as the point with highest speed near the mean position
    center_lat = location_df['latitude'].mean()
    center_lon = location_df['longitude'].mean()
    
    # Find points close to the center (potential start/finish area)
    radius = 0.0001  # Arbitrary small radius, adjust based on track size
    near_center = location_df[(abs(location_df['latitude'] - center_lat) < radius) & 
                            (abs(location_df['longitude'] - center_lon) < radius)]
    
    if not near_center.empty:
        # Use the highest speed point as the start/finish line
        start_point = near_center.loc[near_center['speed'].idxmax()]
        
        # Now find all crossings of this point to identify laps
        laps = []
        lap_start_idx = 0
        crossing_radius = 0.00005  # Smaller radius for crossing detection
        
        for i in range(1, len(location_df)):
            if (abs(location_df.iloc[i]['latitude'] - start_point['latitude']) < crossing_radius and
                abs(location_df.iloc[i]['longitude'] - start_point['longitude']) < crossing_radius):
                # If we're far enough from the last lap start, count as new lap
                if i - lap_start_idx > len(location_df) * 0.05:  # At least 5% of data points between laps
                    lap_data = location_df.iloc[lap_start_idx:i]
                    lap_time = lap_data['seconds_elapsed'].max() - lap_data['seconds_elapsed'].min()
                    avg_speed = lap_data['speed'].mean() * 3.6
                    max_speed = lap_data['speed'].max() * 3.6
                    
                    laps.append({
                        'lap_num': len(laps) + 1,
                        'start_idx': lap_start_idx,
                        'end_idx': i,
                        'lap_time': lap_time,
                        'avg_speed_kmh': avg_speed,
                        'max_speed_kmh': max_speed
                    })
                    lap_start_idx = i
        
        # Create laps DataFrame
        if laps:
            laps_df = pd.DataFrame(laps)
            
            # Display lap times in F1-style table
            print("\n🏁 Lap Analysis 🏁")
            print(f"Total Laps: {len(laps_df)}")
            
            # Format the lap times table like F1 timing screens
            lap_table = laps_df.copy()
            lap_table['lap_time'] = lap_table['lap_time'].round(3).apply(lambda x: f"{int(x//60):01d}:{x%60:06.3f}")
            lap_table['avg_speed_kmh'] = lap_table['avg_speed_kmh'].round(1)
            lap_table['max_speed_kmh'] = lap_table['max_speed_kmh'].round(1)
            
            # Add delta to best lap
            if not lap_table.empty:
                best_lap_idx = laps_df['lap_time'].idxmin()
                best_lap_time = laps_df.loc[best_lap_idx, 'lap_time']
                lap_table['delta'] = laps_df['lap_time'].apply(lambda x: f"+{(x - best_lap_time):.3f}" if x > best_lap_time else "BEST")
                
                # Format the display with emojis for best sectors
                display(HTML(lap_table.style
                            .set_caption("Lap Times")
                            .set_properties(**{'text-align': 'center'})
                            .format({'lap_num': '{:.0f}', 
                                'avg_speed_kmh': '{:.1f}', 
                                'max_speed_kmh': '{:.1f}'})
                            .set_table_attributes('class="dataframe"')
                            .hide(axis='index')  # Updated method to hide index
                            .to_html()))
                
                # Plot lap times comparison
                plt.figure(figsize=(14, 7))
                bar_colors = ['gold' if i == best_lap_idx else 'deepskyblue' for i in range(len(laps_df))]
                plt.bar(lap_table['lap_num'], laps_df['lap_time'], color=bar_colors)
                plt.axhline(y=best_lap_time, color='gold', linestyle='--', alpha=0.7, label=f'Best: {lap_table.loc[best_lap_idx, "lap_time"]}')
                plt.title('Lap Times Comparison', fontsize=16)
                plt.xlabel('Lap Number')
                plt.ylabel('Lap Time (seconds)')
                plt.grid(axis='y', alpha=0.3)
                plt.legend()
                plt.show()

# 4. G-Force Analysis with Telemetry Visualization
plt.figure(figsize=(16, 10))
plt.suptitle('Driver Telemetry', fontsize=20, y=0.98)

# G-Force heatmap
ax1 = plt.subplot(2, 2, 1)
hb = ax1.hexbin(accel_df['lateral_g'], accel_df['longitudinal_g'], 
               gridsize=50, cmap='inferno', 
               extent=[-2, 2, -2, 2])
ax1.set_title('G-Force Distribution', fontsize=14)
ax1.set_xlabel('Lateral G-Force (g)')
ax1.set_ylabel('Longitudinal G-Force (g)')
ax1.grid(True, alpha=0.3)
plt.colorbar(hb, ax=ax1, label='Density')

# Speed histogram
ax2 = plt.subplot(2, 2, 2)
sns.histplot(speed_df['speed_kmh'], bins=30, kde=True, color='cyan', ax=ax2)
ax2.set_title('Speed Distribution', fontsize=14)
ax2.set_xlabel('Speed (km/h)')
ax2.set_ylabel('Frequency')
ax2.grid(True, alpha=0.3)

# Rotational movement (gyroscope)
ax3 = plt.subplot(2, 2, 3)
ax3.plot(gyro_df['seconds_elapsed'], gyro_df['x'], 'r-', alpha=0.7, label='X (Roll)')
ax3.plot(gyro_df['seconds_elapsed'], gyro_df['y'], 'g-', alpha=0.7, label='Y (Pitch)')
ax3.plot(gyro_df['seconds_elapsed'], gyro_df['z'], 'b-', alpha=0.7, label='Z (Yaw)')
ax3.set_title('Rotation Rates', fontsize=14)
ax3.set_xlabel('Time (seconds)')
ax3.set_ylabel('Rotation rate (rad/s)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Speed vs G-Force scatter
ax4 = plt.subplot(2, 2, 4)
sc = ax4.scatter(speed_df['speed_kmh'][:len(accel_df)], accel_df['g_force'][:len(speed_df)], 
                c=accel_df['seconds_elapsed'][:len(speed_df)], cmap='viridis', 
                alpha=0.5, s=15)
ax4.set_title('Speed vs G-Force', fontsize=14)
ax4.set_xlabel('Speed (km/h)')
ax4.set_ylabel('G-Force (g)')
ax4.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax4, label='Time (seconds)')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

# 5. Driver's Line Analysis
if location_df['latitude'].notna().any() and location_df['longitude'].notna().any():
    # Create a visualization showing the racing line colored by speed
    plt.figure(figsize=(14, 12))
    
    # Create a line collection with speed-based coloring
    points = np.array([location_df['longitude'], location_df['latitude']]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    # ... continuing from previous code ...
    
    # Create a colormap
    cmap = plt.get_cmap('turbo')
    norm = plt.Normalize(speed_df['speed_kmh'].min(), speed_df['speed_kmh'].max())
    lc = LineCollection(segments, cmap=cmap, norm=norm, linewidth=2.5)
    lc.set_array(speed_df['speed_kmh'])
    
    plt.gca().add_collection(lc)
    plt.colorbar(lc, label='Speed (km/h)')
    plt.scatter(location_df['longitude'], location_df['latitude'], c='white', s=0.5, alpha=0.2)
    
    plt.title('Racing Line Analysis', fontsize=18)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.axis('equal')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# 6. Sector Analysis - Divide the track into sectors
if location_df['latitude'].notna().any() and location_df['longitude'].notna().any():
    # Let's divide the track into 3 sectors (simplified approach)
    # In a real F1 analysis, sectors would be predefined track segments
    # For our example, we'll just divide by time/distance
    
    total_points = len(location_df)
    sector_size = total_points // 3
    
    location_df['sector'] = 0
    location_df.loc[:sector_size, 'sector'] = 1
    location_df.loc[sector_size:2*sector_size, 'sector'] = 2
    location_df.loc[2*sector_size:, 'sector'] = 3
    
    # Merge sector info with speed data
    speed_df = speed_df.copy()
    speed_df['sector'] = location_df['sector'].values
    
    # Calculate sector statistics
    sector_stats = speed_df.groupby('sector').agg({
        'speed_kmh': ['mean', 'max', 'std'],
        'seconds_elapsed': ['min', 'max']
    })
    
    # Calculate sector times
    sector_stats['sector_time'] = sector_stats[('seconds_elapsed', 'max')] - sector_stats[('seconds_elapsed', 'min')]
    
    # Plot sector analysis
    plt.figure(figsize=(16, 12))
    plt.suptitle('Sector Analysis', fontsize=20, y=0.98)
    
    # Sector map
    ax1 = plt.subplot(2, 2, 1)
    scatter = ax1.scatter(location_df['longitude'], location_df['latitude'], 
                         c=location_df['sector'], cmap='viridis', 
                         s=5, alpha=0.7)
    ax1.set_title('Track Sectors', fontsize=14)
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    ax1.grid(alpha=0.3)
    ax1.axis('equal')
    legend1 = ax1.legend(*scatter.legend_elements(), title="Sector")
    ax1.add_artist(legend1)
    
    # Sector speeds
    ax2 = plt.subplot(2, 2, 2)
    for sector in [1, 2, 3]:
        sector_data = speed_df[speed_df['sector'] == sector]
        ax2.plot(sector_data['seconds_elapsed'], sector_data['speed_kmh'], 
                label=f'Sector {sector}')
    ax2.set_title('Speed by Sector', fontsize=14)
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Speed (km/h)')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # Sector time comparison
    ax3 = plt.subplot(2, 2, 3)
    ax3.bar(['Sector 1', 'Sector 2', 'Sector 3'], 
           sector_stats['sector_time'], 
           color=['#FF9999', '#66B2FF', '#99FF99'])
    for i, v in enumerate(sector_stats['sector_time']):
        ax3.text(i, v/2, f"{v:.1f}s", 
                ha='center', fontsize=12, color='white', fontweight='bold')
    ax3.set_title('Sector Times', fontsize=14)
    ax3.set_ylabel('Time (seconds)')
    ax3.grid(axis='y', alpha=0.3)
    
    # Speed distribution by sector
    ax4 = plt.subplot(2, 2, 4)
    sector_colors = ['#FF9999', '#66B2FF', '#99FF99']
    for sector in [1, 2, 3]:
        sector_data = speed_df[speed_df['sector'] == sector]
        sns.kdeplot(sector_data['speed_kmh'], ax=ax4, 
                   label=f'Sector {sector}', 
                   color=sector_colors[sector-1], fill=True, alpha=0.3)
    ax4.set_title('Speed Distribution by Sector', fontsize=14)
    ax4.set_xlabel('Speed (km/h)')
    ax4.set_ylabel('Density')
    ax4.grid(True, alpha=0.3)
    ax4.legend()
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()
    
    # Print sector statistics in F1-style format
    print("\n🏎️ Sector Analysis 🏎️")
    for sector in [1, 2, 3]:
        mean_speed = sector_stats.loc[sector, ('speed_kmh', 'mean')]
        max_speed = sector_stats.loc[sector, ('speed_kmh', 'max')]
        sector_time = sector_stats.loc[sector, 'sector_time']
        
        # Extract scalar values from Series objects
        if isinstance(mean_speed, pd.Series):
            mean_speed = mean_speed.iloc[0]
        if isinstance(max_speed, pd.Series):
            max_speed = max_speed.iloc[0]
        if isinstance(sector_time, pd.Series):
            sector_time = sector_time.iloc[0]
        
        print(f"Sector {sector}:")
        print(f"  ⏱️ Time: {sector_time:.2f}s")
        print(f"  🚀 Avg Speed: {mean_speed:.1f} km/h")
        print(f"  💨 Max Speed: {max_speed:.1f} km/h")
        print("")

# 7. Advanced G-Force Analysis
# Calculate additional acceleration metrics
accel_df['total_g'] = accel_df['g_force']
accel_df['cornering_g'] = accel_df['lateral_g'].abs()
accel_df['accel_g'] = accel_df['longitudinal_g'].apply(lambda x: max(0, x))
accel_df['brake_g'] = accel_df['longitudinal_g'].apply(lambda x: abs(min(0, x)))

plt.figure(figsize=(16, 12))
plt.suptitle('Advanced G-Force Analysis', fontsize=20, y=0.98)

# Total G-Force
ax1 = plt.subplot(2, 2, 1)
g_threshold_colors = [
    (0, 'blue'),
    (0.5, 'green'),d
    (1.0, 'yellow'),
    (1.5, 'orange'),
    (2.0, 'red'),
    (3.0, 'purple')
]

for i in range(len(g_threshold_colors)-1):
    lower, lower_color = g_threshold_colors[i]
    upper, upper_color = g_threshold_colors[i+1]
    
    mask = (accel_df['total_g'] >= lower) & (accel_df['total_g'] < upper)
    if mask.any():
        ax1.plot(accel_df.loc[mask, 'seconds_elapsed'], 
                accel_df.loc[mask, 'total_g'], 
                '.', markersize=3, color=upper_color, alpha=0.7,
                label=f"{lower:.1f}g-{upper:.1f}g")

ax1.set_title('Total G-Force Over Time', fontsize=14)
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('G-Force (g)')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right')

# Cornering vs Acceleration/Braking balance
ax2 = plt.subplot(2, 2, 2)
cornering_total = accel_df['cornering_g'].sum()
accel_total = accel_df['accel_g'].sum()
brake_total = accel_df['brake_g'].sum()
total = cornering_total + accel_total + brake_total

ax2.pie([cornering_total, accel_total, brake_total], 
       labels=['Cornering', 'Acceleration', 'Braking'],
       autopct='%1.1f%%',
       colors=['#FF9999', '#66B2FF', '#99FF99'],
       startangle=90)
ax2.axis('equal')
ax2.set_title('G-Force Distribution', fontsize=14)

# Time spent at different G-Force levels
ax3 = plt.subplot(2, 2, 3)
g_bins = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
g_labels = ['0-0.5g', '0.5-1.0g', '1.0-1.5g', '1.5-2.0g', '2.0-2.5g', '2.5-3.0g']
g_counts = np.histogram(accel_df['total_g'], bins=g_bins)[0]
g_percentages = g_counts / len(accel_df) * 100

ax3.bar(g_labels, g_percentages, color='turbo')
for i, v in enumerate(g_percentages):
    ax3.text(i, v+0.5, f"{v:.1f}%", ha='center')
ax3.set_title('Time Spent at G-Force Levels', fontsize=14)
ax3.set_xlabel('G-Force Range')
ax3.set_ylabel('Percentage of Time (%)')
ax3.grid(axis='y', alpha=0.3)

# Maximum G-Force by direction
ax4 = plt.subplot(2, 2, 4)
max_lateral_positive = accel_df['lateral_g'].max()
max_lateral_negative = abs(accel_df['lateral_g'].min())
max_accel = accel_df['accel_g'].max()
max_brake = accel_df['brake_g'].max()

categories = ['Left Turn', 'Right Turn', 'Acceleration', 'Braking']
values = [max_lateral_negative, max_lateral_positive, max_accel, max_brake]
colors = ['#FF9999', '#FFCC99', '#99FF99', '#66B2FF']

ax4.bar(categories, values, color=colors)
for i, v in enumerate(values):
    ax4.text(i, v+0.05, f"{v:.2f}g", ha='center')
ax4.set_title('Maximum G-Force by Direction', fontsize=14)
ax4.set_ylabel('G-Force (g)')
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

# 8. Speed Consistency Analysis
if 'speed_kmh' in speed_df.columns and len(speed_df) > 0:
    plt.figure(figsize=(15, 8))
    
    # Calculate rolling stats for speed
    window = 20  # Adjust based on data density
    rolling_mean = speed_df['speed_kmh'].rolling(window=window).mean()
    rolling_std = speed_df['speed_kmh'].rolling(window=window).std()
    
    plt.plot(speed_df['seconds_elapsed'], speed_df['speed_kmh'], 'b-', alpha=0.3, label='Actual Speed')
    plt.plot(speed_df['seconds_elapsed'], rolling_mean, 'r-', linewidth=2, label=f'Average (window={window})')
    plt.fill_between(speed_df['seconds_elapsed'], 
                    rolling_mean - rolling_std, 
                    rolling_mean + rolling_std,
                    color='red', alpha=0.2, label='±1 Std Dev')
    
    plt.title('Speed Consistency Analysis', fontsize=16)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Speed (km/h)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    
    # Calculate variability metrics
    cv = speed_df['speed_kmh'].std() / speed_df['speed_kmh'].mean() * 100
    
    print("\n🚀 Speed Consistency Analysis 🚀")
    print(f"Average Speed: {speed_df['speed_kmh'].mean():.1f} km/h")
    print(f"Standard Deviation: {speed_df['speed_kmh'].std():.1f} km/h")
    print(f"Coefficient of Variation: {cv:.1f}%")
    print(f"Speed Range: {speed_df['speed_kmh'].min():.1f} - {speed_df['speed_kmh'].max():.1f} km/h")
    
# 9. Summary Dashboard with Key Stats
plt.figure(figsize=(12, 8))
plt.suptitle('Session Summary', fontsize=22, y=0.98)

# Create a box at the top for summary stats
props = dict(boxstyle='round', facecolor='black', alpha=0.1)
textstr = '\n'.join((
    f"📊 SESSION STATISTICS",
    f"Session Date: {metadata['recording_time'][0] if not metadata.empty else 'Unknown'}",
    f"Duration: {speed_df['seconds_elapsed'].max():.1f} seconds",
    f"Distance: {speed_df['seconds_elapsed'].max() * speed_df['speed'].mean():.1f} meters",
    f"Avg Speed: {speed_df['speed_kmh'].mean():.1f} km/h",
    f"Max Speed: {speed_df['speed_kmh'].max():.1f} km/h",
    f"Max G-Force: {accel_df['g_force'].max():.2f}g"
))

plt.text(0.5, 0.5, textstr, transform=plt.gcf().transFigure, fontsize=14,
        verticalalignment='center', horizontalalignment='center', bbox=props)

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

print("\n🏁 Analysis Complete! 🏁")
conn.close()


In [ ]:
# Simple map of location data using matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Check if we have location data
if location_df['latitude'].notna().any() and location_df['longitude'].notna().any():
    # Create a clean figure
    plt.figure(figsize=(12, 10))
    
    # Create a colormap for speed
    speeds = location_df['speed'] * 3.6  # Convert to km/h
    
    # Create scatter plot with speed coloring
    sc = plt.scatter(
        location_df['longitude'], 
        location_df['latitude'],
        c=speeds,
        cmap='viridis',
        s=15,  # Marker size
        alpha=0.8,  # Transparency
        edgecolors='none'
    )
    
    # Add a colorbar
    cbar = plt.colorbar(sc)
    cbar.set_label('Speed (km/h)', fontsize=12)
    
    # Add track points connecting lines to show the path
    plt.plot(
        location_df['longitude'],
        location_df['latitude'],
        '-',
        color='white',
        alpha=0.3,
        linewidth=1
    )
    
    # Mark start point with a green dot
    plt.plot(
        location_df['longitude'].iloc[0],
        location_df['latitude'].iloc[0],
        'go',  # Green circle
        markersize=12,
        label='Start'
    )
    
    # Mark end point with a red dot
    plt.plot(
        location_df['longitude'].iloc[-1],
        location_df['latitude'].iloc[-1],
        'ro',  # Red circle
        markersize=12,
        label='End'
    )
    
    # Adjust plot aesthetics
    plt.title('Go-Kart Track Map', fontsize=16)
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.grid(alpha=0.3)
    plt.axis('equal')  # Equal aspect ratio
    plt.legend(loc='best')
    
    # Add a text box with basic stats
    max_speed = speeds.max()
    avg_speed = speeds.mean()
    distance = location_df['speed'].mean() * (location_df['seconds_elapsed'].max() - location_df['seconds_elapsed'].min())
    
    stats_text = (
        f"Max Speed: {max_speed:.1f} km/h\n"
        f"Avg Speed: {avg_speed:.1f} km/h\n"
        f"Est. Distance: {distance:.0f} m"
    )
    
    plt.annotate(
        stats_text,
        xy=(0.02, 0.02),
        xycoords='axes fraction',
        bbox=dict(boxstyle="round,pad=0.5", fc="black", ec="cyan", alpha=0.7),
        color='white',
        fontsize=10
    )
    
    plt.tight_layout()
    plt.show()
    
    # Additionally, create a heatmap-style track map showing where the driver spends most time
    plt.figure(figsize=(12, 10))
    
    # Use hexbin for density visualization
    hb = plt.hexbin(
        location_df['longitude'],
        location_df['latitude'],
        gridsize=50,
        cmap='inferno',
        alpha=0.7
    )
    
    # Add a colorbar
    cbar = plt.colorbar(hb)
    cbar.set_label('Density (points count)', fontsize=12)
    
    # Overlay the track path as a semi-transparent line
    plt.plot(
        location_df['longitude'],
        location_df['latitude'],
        '-',
        color='white',
        alpha=0.3,
        linewidth=1
    )
    
    plt.title('Track Heatmap - Time Spent in Different Sections', fontsize=16)
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.grid(alpha=0.3)
    plt.axis('equal')
    
    plt.tight_layout()
    plt.show()
else:
    print("No location data available to create a map.")

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load the location data from CSV
location_df = pd.read_csv('/Users/nathanverrill/Boschertown/boschertown-1/Location.csv')

# Load the geofence from GeoJSON
geofence = gpd.read_file('/Users/nathanverrill/Boschertown/boschertown_geofence.geojson')

# Create geometry points from latitude and longitude columns
# Assuming the CSV has columns named 'latitude' and 'longitude'
# If they have different names, please adjust accordingly
geometry = [Point(xy) for xy in zip(location_df['longitude'], location_df['latitude'])]
location_gdf = gpd.GeoDataFrame(location_df, geometry=geometry, crs="EPSG:4326")

# Make sure the CRS of both GeoDataFrames match
if geofence.crs != location_gdf.crs:
    geofence = geofence.to_crs(location_gdf.crs)

# Filter to keep ONLY the points that are INSIDE the geofence
polygon = geofence.geometry.iloc[0]
filtered_gdf = location_gdf[location_gdf.geometry.within(polygon)]

# Drop the geometry column as it's not needed in the output CSV
filtered_df = filtered_gdf.drop(columns=['geometry'])

# Save the filtered data to a new CSV file
filtered_df.to_csv('./location_filtered.csv', index=False)

print(f"Original data points: {len(location_df)}")
print(f"Points inside the geofence: {len(filtered_df)}")
print(f"Points filtered out (outside geofence): {len(location_df) - len(filtered_df)}")
print(f"Filtered data saved to ./location_filtered.csv")

In [ ]:
import pandas as pd
from datetime import datetime

# Load the location data
file_path = '/Users/nathanverrill/Boschertown/boschertown-1/Location.csv'
location_df = pd.read_csv(file_path)

# Trim the data to keep only rows between 88 and 1046 seconds_elapsed
# Using .copy() to avoid SettingWithCopyWarning
trimmed_df = location_df[(location_df['seconds_elapsed'] >= 88) & 
                         (location_df['seconds_elapsed'] <= 1046)].copy()

# Convert nanosecond timestamp to datetime
def convert_to_datetime(ns_timestamp):
    # Convert nanoseconds to seconds
    seconds = ns_timestamp / 1e9
    # Convert to datetime
    dt = datetime.fromtimestamp(seconds)
    return dt

# Add timestamp column with readable datetime
trimmed_df['timestamp'] = trimmed_df['time'].apply(convert_to_datetime)

# Display info about the original and trimmed data
print(f"Original data: {len(location_df)} rows")
print(f"Trimmed data: {len(trimmed_df)} rows")

# Save the trimmed data to a new file
trimmed_file_path = './Location_trimmed.csv'
trimmed_df.to_csv(trimmed_file_path, index=False)
print(f"Trimmed data saved to {trimmed_file_path}")

# Display the first few rows of the trimmed data
print("\nFirst few rows of trimmed data with timestamp:")
print(trimmed_df[['seconds_elapsed', 'time', 'timestamp']].head())

In [ ]:
import sqlite3
import os

def list_tables_in_sqlite_db(db_path):
    """
    Lists all tables in the specified SQLite database file.
    """
    if not os.path.exists(db_path):
        print(f"Error: Database file '{db_path}' not found.")
        return
    
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Query to get all table names
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        
        if tables:
            print(f"Tables in {db_path}:")
            for i, table_name in enumerate(tables):
                print(f"{i+1}. {table_name[0]}")
        else:
            print(f"No tables found in {db_path}")
        
        # Close the connection
        conn.close()
        
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")

if __name__ == "__main__":
    db_path = "./b.sqlite"
    list_tables_in_sqlite_db(db_path)

In [ ]:
import sqlite3
import os

def list_location_rows(db_path, limit=10):
    """
    Lists the first n rows from the Location table in the specified SQLite database.
    """
    if not os.path.exists(db_path):
        print(f"Error: Database file '{db_path}' not found.")
        return
    
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_path)
        conn.row_factory = sqlite3.Row  # This enables column access by name
        cursor = conn.cursor()
        
        # First check if the Location table exists
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='Location';")
        if not cursor.fetchone():
            print("Error: 'Location' table doesn't exist in the database.")
            conn.close()
            return
        
        # Get column names
        cursor.execute("PRAGMA table_info(Location)")
        columns = [column[1] for column in cursor.fetchall()]
        
        # Query to get the first n rows
        cursor.execute(f"SELECT * FROM Location LIMIT {limit}")
        rows = cursor.fetchall()
        
        if rows:
            print(f"First {limit} rows from Location table:")
            print("-" * 80)
            # Print column headers
            print(" | ".join(columns))
            print("-" * 80)
            
            # Print rows
            for row in rows:
                row_data = [str(row[col]) for col in columns]
                print(" | ".join(row_data))
        else:
            print("No data found in the Location table.")
        
        # Close the connection
        conn.close()
        
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
f
if __name__ == "__main__":
    db_path = "./b.sqlite"
    list_location_rows(db_path)

In [ ]:
import sqlite3
import os
import sys

def trim_tables_by_seconds_elapsed(db_path, min_seconds=88, max_seconds=800):
    """
    Trims all tables in the database to keep only rows where 'seconds_elapsed' 
    is between the specified min and max values.
    """
    if not os.path.exists(db_path):
        print(f"Error: Database file '{db_path}' not found.")
        return
    
    # Create a backup of the database before making changes
    backup_path = f"{db_path}.backup"
    try:
        import shutil
        shutil.copy2(db_path, backup_path)
        print(f"Backup created at {backup_path}")
    except Exception as e:
        print(f"Warning: Could not create backup: {e}")
        response = input("Continue without backup? (y/n): ")
        if response.lower() != 'y':
            print("Operation cancelled.")
            return
    
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Get all table names
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        
        tables_trimmed = 0
        rows_deleted = 0
        tables_with_seconds_elapsed = []
        
        # Process each table
        for table in tables:
            table_name = table[0]
            
            # Check if the table has a seconds_elapsed column
            cursor.execute(f"PRAGMA table_info({table_name})")
            columns = cursor.fetchall()
            has_seconds_elapsed = any(col[1].lower() == 'seconds_elapsed' for col in columns)
            
            if has_seconds_elapsed:
                tables_with_seconds_elapsed.append(table_name)
                
                # Count rows before deletion
                cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
                before_count = cursor.fetchone()[0]
                
                # Delete rows outside the specified range
                cursor.execute(f"""
                    DELETE FROM {table_name} 
                    WHERE seconds_elapsed < {min_seconds} OR seconds_elapsed > {max_seconds}
                """)
                
                # Count rows after deletion
                cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
                after_count = cursor.fetchone()[0]
                
                deleted_in_table = before_count - after_count
                rows_deleted += deleted_in_table
                tables_trimmed += 1
                
                print(f"Table '{table_name}': {deleted_in_table} rows deleted, {after_count} rows remaining")
        
        # Commit changes
        conn.commit()
        
        # Report results
        if tables_with_seconds_elapsed:
            print(f"\nTrimming complete. {rows_deleted} total rows deleted from {tables_trimmed} tables.")
            print(f"Tables affected: {', '.join(tables_with_seconds_elapsed)}")
        else:
            print("\nNo tables found with a 'seconds_elapsed' column.")
        
        # Vacuum the database to reclaim space
        cursor.execute("VACUUM")
        conn.close()
        
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        print("Changes rolled back.")
        conn.rollback()
        conn.close()

if __name__ == "__main__":
    db_path = "./b.sqlite"
    
    print("This script will trim data in all tables to keep only rows where seconds_elapsed is between 88 and 1046.")
    print("A backup will be created before any changes are made.")
    response = input("Continue? (y/n): ")
    
    if response.lower() == 'y':
        trim_tables_by_seconds_elapsed(db_path)
    else:
        print("Operation cancelled.")

In [ ]:
# SQLite Database Explorer for Jupyter Notebook - Fixed Version with all indexing issues resolved
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display, HTML

# Set plot style
plt.style.use('ggplot')
sns.set(font_scale=1.2)

# Connect to the database
db_path = "./b.sqlite"
conn = sqlite3.connect(db_path)

# Function to get all table names
def get_tables():
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [table[0] for table in cursor.fetchall()]
    return tables

# Function to get table info
def get_table_info(table_name):
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name})")
    return cursor.fetchall()

# Function to get row count for a table
def get_row_count(table_name):
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    return cursor.fetchone()[0]

# Function to check if column exists and is numeric
def is_numeric_column(table_name, column_name):
    try:
        test_df = pd.read_sql_query(f"SELECT {column_name} FROM {table_name} LIMIT 1", conn)
        return pd.api.types.is_numeric_dtype(test_df.dtypes.iloc[0])
    except:
        return False

# Function to display basic statistics for a table
def display_table_stats(table_name):
    columns = [col[1] for col in get_table_info(table_name)]
    row_count = get_row_count(table_name)
    
    print(f"## {table_name} Table")
    print(f"- Number of rows: {row_count}")
    print(f"- Columns: {', '.join(columns)}")
    
    # Get sample data
    df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
    print("\n### Sample Data:")
    display(df)
    
    # Check if seconds_elapsed exists and get min/max
    if 'seconds_elapsed' in columns:
        time_stats = pd.read_sql_query(
            f"SELECT MIN(seconds_elapsed) as min_time, MAX(seconds_elapsed) as max_time FROM {table_name}", 
            conn
        )
        print(f"\n### Time Range:")
        # Handle potential NULL values using .iloc accessor
        min_time = time_stats['min_time'].iloc[0]
        max_time = time_stats['max_time'].iloc[0]
        
        if min_time is not None:
            print(f"- Min seconds_elapsed: {min_time:.2f}")
        else:
            print("- Min seconds_elapsed: None")
            
        if max_time is not None:
            print(f"- Max seconds_elapsed: {max_time:.2f}")
        else:
            print("- Max seconds_elapsed: None")
    
    # Get numeric columns
    numeric_cols = [col for col in columns if is_numeric_column(table_name, col)]
    
    if numeric_cols and len(numeric_cols) <= 10:  # Only if there are reasonable number of numeric columns
        print("\n### Numerical Column Statistics:")
        stats_query = f"SELECT "
        for col in numeric_cols:
            stats_query += f"AVG({col}) as {col}_avg, MIN({col}) as {col}_min, MAX({col}) as {col}_max, "
        stats_query = stats_query.rstrip(", ")
        stats_query += f" FROM {table_name}"
        
        try:
            stats = pd.read_sql_query(stats_query, conn)
            display(stats)
        except Exception as e:
            print(f"Error calculating statistics: {e}")

# Function to plot data from a table
def plot_table_data(table_name, max_points=1000):
    columns = [col[1] for col in get_table_info(table_name)]
    
    # Only proceed if the table has seconds_elapsed
    if 'seconds_elapsed' not in columns:
        print(f"Table {table_name} doesn't have seconds_elapsed column, skipping plots")
        return
    
    # Get numeric columns excluding seconds_elapsed
    numeric_cols = [col for col in columns if col != 'seconds_elapsed' and is_numeric_column(table_name, col)]
    
    # If we have too many numeric columns, limit to first few
    if len(numeric_cols) > 4:
        numeric_cols = numeric_cols[:4]
        print(f"Too many numeric columns, plotting only the first 4: {numeric_cols}")
    
    if not numeric_cols:
        print(f"No suitable numeric columns found in {table_name} for plotting")
        return
    
    # Sample data for plotting to avoid overloading
    row_count = get_row_count(table_name)
    if row_count > max_points:
        sample_rate = int(row_count / max_points)
        query = f"SELECT seconds_elapsed, {', '.join(numeric_cols)} FROM {table_name} WHERE rowid % {sample_rate} = 0"
    else:
        query = f"SELECT seconds_elapsed, {', '.join(numeric_cols)} FROM {table_name}"
    
    try:
        df = pd.read_sql_query(query, conn)
        
        # Skip if no data
        if df.empty:
            print(f"No data available for plotting in {table_name}")
            return
            
        # Plot time series
        fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(12, 4*len(numeric_cols)))
        if len(numeric_cols) == 1:
            axes = [axes]  # Make axes iterable if there's only one subplot
        
        for i, col in enumerate(numeric_cols):
            # Filter out None/NaN values for plotting
            plot_data = df.dropna(subset=['seconds_elapsed', col])
            if len(plot_data) > 0:
                axes[i].plot(plot_data['seconds_elapsed'], plot_data[col], 'b-', alpha=0.7)
                axes[i].set_title(f"{table_name}: {col} over Time")
                axes[i].set_xlabel('Seconds Elapsed')
                axes[i].set_ylabel(col)
                axes[i].grid(True)
            else:
                axes[i].text(0.5, 0.5, f"No valid data for {col}", 
                             horizontalalignment='center',
                             verticalalignment='center',
                             transform=axes[i].transAxes)
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error plotting {table_name}: {e}")

# Main execution
print(f"# SQLite Database Explorer: {db_path}")
tables = get_tables()
print(f"Found {len(tables)} tables in the database.")

# Display table list with row counts
table_counts = []
for table in tables:
    count = get_row_count(table)
    table_counts.append((table, count))

# Sort by row count descending
table_counts.sort(key=lambda x: x[1], reverse=True)

print("\n## Tables by Row Count")
for table, count in table_counts:
    print(f"- {table}: {count} rows")

# Display details for each table
for table in tables:
    print("\n" + "="*80)
    try:
        display_table_stats(table)
        plot_table_data(table)
    except Exception as e:
        print(f"Error processing table {table}: {e}")
    print("="*80)

# Close connection
conn.close()

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Connect to the database
db_path = "./b.sqlite"
conn = sqlite3.connect(db_path)

# Check if all requested columns exist
try:
    cursor = conn.cursor()
    cursor.execute("PRAGMA table_info(Orientation)")
    columns_info = cursor.fetchall()
    table_columns = [col[1] for col in columns_info]
    
    requested_columns = ['yaw', 'qx', 'qz', 'roll', 'qw', 'qy', 'pitch']
    available_columns = [col for col in requested_columns if col in table_columns]
    
    if not available_columns:
        print("None of the requested columns exist in the Orientation table.")
        conn.close()
        exit()
    
    missing_columns = [col for col in requested_columns if col not in table_columns]
    if missing_columns:
        print(f"Warning: The following columns are not in the Orientation table: {', '.join(missing_columns)}")
    
    # Get the data from the Orientation table
    query = f"""
    SELECT seconds_elapsed, {', '.join(available_columns)}
    FROM Orientation 
    ORDER BY seconds_elapsed
    """
    
    # Load the data
    df = pd.read_sql_query(query, conn)
    
    # Check if we have data
    if df.empty:
        print("No data found in the Orientation table.")
    else:
        # Print basic statistics
        print("Statistics for orientation components:")
        print(f"Number of data points: {len(df)}")
        print(f"Time range: {df['seconds_elapsed'].min():.2f} to {df['seconds_elapsed'].max():.2f} seconds")
        
        # Calculate a reasonable window size for the rolling average
        window_size = min(51, len(df) // 10)  # Adaptive window size based on data length
        if window_size % 2 == 0:  # Ensure window size is odd
            window_size += 1
            
        # Create individual plots for each component
        for component in available_columns:
            print(f"\n{component} statistics:")
            print(f"Range: {df[component].min():.4f} to {df[component].max():.4f}")
            print(f"Mean: {df[component].mean():.4f}")
            print(f"Standard deviation: {df[component].std():.4f}")
            
            plt.figure(figsize=(14, 7))
            
            # Choose color based on component type
            color = 'blue'  # Default
            if component.startswith('q'):
                color = 'purple'  # Quaternion components
            elif component in ['roll', 'pitch', 'yaw']:
                color = 'green'  # Euler angles
                
            # Plot the raw data
            plt.plot(df['seconds_elapsed'], df[component], color=color, alpha=0.6, 
                     label=f'Raw {component} values')
            
            # Add rolling average if we have enough data
            if len(df) > window_size * 2:
                rolling_avg = df[component].rolling(window=window_size, center=True).mean()
                plt.plot(df['seconds_elapsed'], rolling_avg, color='black', linewidth=2, 
                        label=f'{window_size}-point rolling average')
            
            # Add horizontal line at zero for reference
            plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
            
            # Set plot labels and styling
            plt.title(f'{component.upper()} Values Over Time', fontsize=16)
            plt.xlabel('Time (seconds)', fontsize=14)
            plt.ylabel(f'{component} Value', fontsize=14)
            plt.grid(True, alpha=0.3)
            plt.legend(loc='upper right')
            
            # Add context information based on the component type
            if component.startswith('q'):
                plt.figtext(0.5, 0.01, 
                          f"Note: {component} is a quaternion component used to represent 3D orientation",
                          ha='center', fontsize=10, style='italic')
            elif component in ['roll', 'pitch', 'yaw']:
                plt.figtext(0.5, 0.01, 
                          f"Note: {component} is an Euler angle representing rotation around the " + 
                          ("X-axis (lateral)" if component == 'roll' else 
                           "Y-axis (longitudinal)" if component == 'pitch' else "Z-axis (vertical)"),
                          ha='center', fontsize=10, style='italic')
            
            plt.tight_layout()
            plt.show()
        
        # Create a combined plot for all quaternion components
        q_components = [col for col in available_columns if col.startswith('q')]
        if len(q_components) > 1:
            plt.figure(figsize=(14, 7))
            colors = ['blue', 'green', 'red', 'purple']
            
            for i, component in enumerate(q_components):
                plt.plot(df['seconds_elapsed'], df[component], color=colors[i % len(colors)], 
                         alpha=0.7, label=component)
            
            plt.title('Combined Quaternion Components Over Time', fontsize=16)
            plt.xlabel('Time (seconds)', fontsize=14)
            plt.ylabel('Value', fontsize=14)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()
            plt.show()
        
        # Create a combined plot for Euler angles (roll, pitch, yaw)
        euler_angles = [col for col in available_columns if col in ['roll', 'pitch', 'yaw']]
        if len(euler_angles) > 1:
            plt.figure(figsize=(14, 7))
            colors = ['blue', 'green', 'red']
            
            for i, component in enumerate(euler_angles):
                plt.plot(df['seconds_elapsed'], df[component], color=colors[i % len(colors)], 
                         alpha=0.7, label=component)
            
            plt.title('Combined Euler Angles Over Time', fontsize=16)
            plt.xlabel('Time (seconds)', fontsize=14)
            plt.ylabel('Angle (degrees or radians)', fontsize=14)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()
            plt.show()
            
except Exception as e:
    print(f"Error analyzing orientation data: {e}")
finally:
    # Close connection
    conn.close()

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
from scipy.interpolate import interp1d

# Connect to the database
db_path = "./b.sqlite"
conn = sqlite3.connect(db_path)

try:
    # 1. Load the Location data (lower frequency)
    location_df = pd.read_sql_query("""
        SELECT time, seconds_elapsed, latitude, longitude, altitude, speed
        FROM Location
        ORDER BY seconds_elapsed
    """, conn)
    
    # 2. Load the Orientation data (higher frequency)
    orientation_df = pd.read_sql_query("""
        SELECT time, seconds_elapsed, qx, qy, qz, qw, pitch, roll, yaw
        FROM Orientation
        ORDER BY seconds_elapsed
    """, conn)
    
    print(f"Location data: {len(location_df)} points")
    print(f"Orientation data: {len(orientation_df)} points")
    
    # 3. Determine the common time range
    loc_time_min = location_df['seconds_elapsed'].min()
    loc_time_max = location_df['seconds_elapsed'].max()
    ori_time_min = orientation_df['seconds_elapsed'].min()
    ori_time_max = orientation_df['seconds_elapsed'].max()
    
    common_min = max(loc_time_min, ori_time_min)
    common_max = min(loc_time_max, ori_time_max)
    
    print(f"Common time range: {common_min:.2f} to {common_max:.2f} seconds")
    
    # 4. Filter data to common time range
    location_df = location_df[(location_df['seconds_elapsed'] >= common_min) & 
                             (location_df['seconds_elapsed'] <= common_max)]
    orientation_df = orientation_df[(orientation_df['seconds_elapsed'] >= common_min) & 
                                  (orientation_df['seconds_elapsed'] <= common_max)]
    
    # 5. Create interpolation functions for Location data
    location_interp = {}
    for col in location_df.columns:
        if col not in ['time', 'seconds_elapsed']:
            location_interp[col] = interp1d(
                location_df['seconds_elapsed'], 
                location_df[col], 
                kind='linear', 
                bounds_error=False,
                fill_value=(location_df[col].iloc[0], location_df[col].iloc[-1])
            )
    
    # 6. Choose which timestamps to use for the combined dataset
    # Option 1: Use orientation timestamps (preserves higher frequency)
    timestamps = orientation_df['seconds_elapsed']
    
    # 7. Create the combined dataset
    combined_data = []
    
    # For each timestamp in our chosen set
    for ts in timestamps:
        row = {'seconds_elapsed': ts}
        
        # Add orientation data from the nearest orientation timestamp
        ori_idx = (orientation_df['seconds_elapsed'] - ts).abs().idxmin()
        for col in orientation_df.columns:
            if col != 'seconds_elapsed' and col != 'time':
                row[col] = orientation_df.loc[ori_idx, col]
        
        # Add interpolated location data
        for col, interp_func in location_interp.items():
            row[col] = float(interp_func(ts))
        
        combined_data.append(row)
    
    # Create the combined dataframe
    combined_df = pd.DataFrame(combined_data)
    
    # 8. Add a timestamp column in a format Kepler can use
    # Convert seconds_elapsed to timestamp by assuming a base time
    combined_df['timestamp'] = pd.to_datetime('2023-01-01') + pd.to_timedelta(combined_df['seconds_elapsed'], unit='s')
    
    # 9. Save to CSV
    output_file = 'combined_location_orientation.csv'
    combined_df.to_csv(output_file, index=False)
    
    print(f"Combined data saved to {output_file}")
    print(f"Total rows: {len(combined_df)}")
    print("\nFirst few rows:")
    print(combined_df.head())
    
    print("\nColumns available:")
    for col in combined_df.columns:
        print(f"- {col}")
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()
finally:
    conn.close()

In [ ]:
import sqlite3
import pandas as pd

# Connect to the database
db_path = "./b.sqlite"
conn = sqlite3.connect(db_path)

try:
    # Query to get all speed values
    speed_query = """
        SELECT seconds_elapsed, speed
        FROM Location
        WHERE speed IS NOT NULL
        ORDER BY speed DESC
    """
    
    speed_df = pd.read_sql_query(speed_query, conn)
    
    if speed_df.empty:
        print("No speed data found in the database.")
    else:
        # Find the maximum speed
        max_speed_ms = speed_df['speed'].max()
        max_speed_kmh = max_speed_ms * 3.6  # Convert m/s to km/h
        max_speed_mph = max_speed_ms * 2.23694  # Convert m/s to mph
        
        # Get the time when the maximum speed occurred
        max_speed_time = speed_df.iloc[0]['seconds_elapsed']
        
        # Get additional details about the maximum speed point
        max_speed_details_query = f"""
            SELECT seconds_elapsed, speed, latitude, longitude, altitude
            FROM Location
            WHERE seconds_elapsed = {max_speed_time}
        """
        
        max_speed_details = pd.read_sql_query(max_speed_details_query, conn)
        
        print("\n===== TOP SPEED ANALYSIS =====\n")
        print(f"Top speed: {max_speed_ms:.2f} m/s ({max_speed_kmh:.2f} km/h, {max_speed_mph:.2f} mph)")
        print(f"Recorded at {max_speed_time:.2f} seconds elapsed")
        
        if not max_speed_details.empty:
            lat = max_speed_details.iloc[0]['latitude']
            lon = max_speed_details.iloc[0]['longitude']
            alt = max_speed_details.iloc[0]['altitude'] if 'altitude' in max_speed_details.columns else "N/A"
            print(f"Location: Latitude {lat:.6f}, Longitude {lon:.6f}, Altitude {alt}")
            print(f"Google Maps link: https://www.google.com/maps?q={lat},{lon}")
        
        # Get speed statistics
        print("\n----- Speed Statistics -----")
        print(f"Average speed: {speed_df['speed'].mean() * 3.6:.2f} km/h")
        print(f"Median speed: {speed_df['speed'].median() * 3.6:.2f} km/h")
        print(f"Min speed: {speed_df['speed'].min() * 3.6:.2f} km/h")
        
        # Get top 5 speeds
        print("\n----- Top 5 Speeds -----")
        top_speeds = speed_df.head(5)
        for i, (_, row) in enumerate(top_speeds.iterrows()):
            print(f"{i+1}. {row['speed'] * 3.6:.2f} km/h at {row['seconds_elapsed']:.2f} seconds")
        
        # Create a speed histogram
        print("\n----- Speed Distribution -----")
        
        # Convert to km/h for the histogram
        speed_km_h = speed_df['speed'] * 3.6
        
        # Calculate percentiles
        p10 = speed_km_h.quantile(0.10)
        p25 = speed_km_h.quantile(0.25)
        p50 = speed_km_h.quantile(0.50)
        p75 = speed_km_h.quantile(0.75)
        p90 = speed_km_h.quantile(0.90)
        
        print(f"10th percentile: {p10:.2f} km/h")
        print(f"25th percentile: {p25:.2f} km/h")
        print(f"50th percentile (median): {p50:.2f} km/h")
        print(f"75th percentile: {p75:.2f} km/h")
        print(f"90th percentile: {p90:.2f} km/h")
        
        # Optional: Plot speed over time
        try:
            import matplotlib.pyplot as plt
            
            # Get data sorted by time for plotting
            time_query = """
                SELECT seconds_elapsed, speed
                FROM Location
                WHERE speed IS NOT NULL
                ORDER BY seconds_elapsed
            """
            
            time_df = pd.read_sql_query(time_query, conn)
            
            plt.figure(figsize=(12, 6))
            plt.plot(time_df['seconds_elapsed'], time_df['speed'] * 3.6, 'b-')
            plt.axhline(y=max_speed_kmh, color='r', linestyle='--', 
                       label=f'Top Speed: {max_speed_kmh:.2f} km/h')
            
            plt.title('Speed Over Time')
            plt.xlabel('Seconds Elapsed')
            plt.ylabel('Speed (km/h)')
            plt.grid(True, alpha=0.3)
            plt.legend()
            
            # Mark the top speed point
            plt.plot(max_speed_time, max_speed_kmh, 'ro', markersize=8)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Could not create speed plot: {e}")
        
except Exception as e:
    print(f"Error analyzing speed data: {e}")
finally:
    conn.close()

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# Connect to the database
db_path = "./b.sqlite"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

try:
    # Get all table names from the database
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [table[0] for table in cursor.fetchall()]
    
    print(f"Found {len(tables)} tables in the database.")
    
    # Dictionary to store results
    results = {}
    
    # For each table, analyze the data collection frequency
    for table in tables:
        print(f"\nAnalyzing table: {table}")
        
        # Check if the table has a seconds_elapsed column
        cursor.execute(f"PRAGMA table_info({table})")
        columns = [col[1] for col in cursor.fetchall()]
        
        if 'seconds_elapsed' not in columns:
            print(f"  Table {table} does not have a seconds_elapsed column, skipping.")
            continue
        
        # Get the count of rows
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        row_count = cursor.fetchone()[0]
        
        if row_count == 0:
            print(f"  Table {table} is empty, skipping.")
            continue
        
        # Get the time range
        cursor.execute(f"SELECT MIN(seconds_elapsed), MAX(seconds_elapsed) FROM {table}")
        min_time, max_time = cursor.fetchone()
        time_range = max_time - min_time
        
        # Calculate the overall frequency (rows per second)
        overall_frequency = row_count / time_range if time_range > 0 else 0
        
        # Get all timestamps to analyze actual intervals
        cursor.execute(f"SELECT seconds_elapsed FROM {table} ORDER BY seconds_elapsed")
        timestamps = [row[0] for row in cursor.fetchall()]
        
        # Calculate intervals between consecutive records
        intervals = np.diff(timestamps)
        
        # Remove outliers for more accurate frequency calculation
        if len(intervals) > 10:  # Only if we have enough data points
            q1, q3 = np.percentile(intervals, [25, 75])
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            filtered_intervals = intervals[(intervals >= lower_bound) & (intervals <= upper_bound)]
        else:
            filtered_intervals = intervals
        
        # Calculate median interval and frequency
        median_interval = np.median(filtered_intervals) if len(filtered_intervals) > 0 else 0
        median_frequency = 1.0 / median_interval if median_interval > 0 else 0
        
        # Calculate common intervals
        interval_counts = defaultdict(int)
        for interval in intervals:
            # Round to 4 decimal places for grouping
            rounded = round(interval, 4)
            interval_counts[rounded] += 1
        
        # Get the most common intervals
        most_common = sorted(interval_counts.items(), key=lambda x: x[1], reverse=True)[:3]
        
        # Store the results
        results[table] = {
            'row_count': row_count,
            'time_range': time_range,
            'overall_frequency': overall_frequency,
            'median_interval': median_interval,
            'median_frequency': median_frequency,
            'common_intervals': most_common
        }
        
        print(f"  Rows: {row_count}")
        print(f"  Time range: {min_time:.2f}s to {max_time:.2f}s ({time_range:.2f}s)")
        print(f"  Overall frequency: {overall_frequency:.2f} Hz (records per second)")
        print(f"  Median interval between records: {median_interval:.4f}s")
        print(f"  Median frequency: {median_frequency:.2f} Hz")
        
        print("  Most common intervals:")
        for interval, count in most_common:
            freq = 1.0 / interval if interval > 0 else 0
            percentage = (count / len(intervals)) * 100
            print(f"    {interval:.4f}s ({freq:.2f} Hz): {count} occurrences ({percentage:.1f}%)")
    
    # Print summary table
    print("\n\n===== SUMMARY OF DATA COLLECTION FREQUENCIES =====")
    summary_data = []
    
    for table, data in results.items():
        summary_data.append({
            'Table': table,
            'Records': data['row_count'],
            'Duration (s)': round(data['time_range'], 2),
            'Frequency (Hz)': round(data['median_frequency'], 2),
            'Interval (s)': round(data['median_interval'], 4)
        })
    
    # Sort by frequency (highest first)
    summary_df = pd.DataFrame(summary_data).sort_values('Frequency (Hz)', ascending=False)
    print(summary_df.to_string(index=False))
    
    # Create bar chart comparing frequencies
    plt.figure(figsize=(12, 6))
    plt.bar(summary_df['Table'], summary_df['Frequency (Hz)'])
    plt.xticks(rotation=45, ha='right')
    plt.title('Data Collection Frequency by Table')
    plt.xlabel('Table')
    plt.ylabel('Frequency (Hz)')
    plt.tight_layout()
    plt.grid(axis='y', alpha=0.3)
    plt.show()
    
    # Create a comparison of record counts
    plt.figure(figsize=(12, 6))
    plt.bar(summary_df['Table'], summary_df['Records'])
    plt.xticks(rotation=45, ha='right')
    plt.title('Number of Records by Table')
    plt.xlabel('Table')
    plt.ylabel('Number of Records')
    plt.tight_layout()
    plt.grid(axis='y', alpha=0.3)
    plt.show()
    
except Exception as e:
    print(f"Error analyzing data frequencies: {e}")
    import traceback
    traceback.print_exc()
finally:
    conn.close()

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Connect to the database
db_path = "./b.sqlite"
conn = sqlite3.connect(db_path)

try:
    # Load the Location table data
    location_df = pd.read_sql_query("""
        SELECT seconds_elapsed, latitude, longitude, speed, horizontalAccuracy
        FROM Location
        ORDER BY seconds_elapsed
    """, conn)
    
    print(f"Location table contains {len(location_df)} records")
    
    if location_df.empty:
        print("No data found in Location table.")
    else:
        # Calculate basic time range statistics
        time_min = location_df['seconds_elapsed'].min()
        time_max = location_df['seconds_elapsed'].max()
        time_range = time_max - time_min
        
        print(f"Time range: {time_min:.2f}s to {time_max:.2f}s ({time_range:.2f} seconds)")
        
        # Overall average frequency
        overall_frequency = len(location_df) / time_range
        print(f"Overall average frequency: {overall_frequency:.4f} Hz ({overall_frequency * 60:.2f} records/minute)")
        
        # Calculate intervals between consecutive records
        intervals = np.diff(location_df['seconds_elapsed'].values)
        
        # Basic interval statistics
        min_interval = np.min(intervals)
        max_interval = np.max(intervals)
        mean_interval = np.mean(intervals)
        median_interval = np.median(intervals)
        
        print("\n----- Interval Statistics -----")
        print(f"Minimum interval: {min_interval:.4f}s ({1/min_interval:.2f} Hz)")
        print(f"Maximum interval: {max_interval:.4f}s ({1/max_interval:.2f} Hz if continuous)")
        print(f"Mean interval: {mean_interval:.4f}s ({1/mean_interval:.2f} Hz)")
        print(f"Median interval: {median_interval:.4f}s ({1/median_interval:.2f} Hz)")
        
        # Identify potential GPS signal loss periods (unusually long intervals)
        # Define a threshold as 3x the median interval
        threshold = median_interval * 3
        gaps = intervals[intervals > threshold]
        gap_indices = np.where(intervals > threshold)[0]
        
        if len(gaps) > 0:
            print(f"\n----- Detected {len(gaps)} potential GPS signal loss periods -----")
            for i, gap_idx in enumerate(gap_indices):
                gap_start_time = location_df['seconds_elapsed'].iloc[gap_idx]
                gap_end_time = location_df['seconds_elapsed'].iloc[gap_idx + 1]
                gap_duration = gaps[i]
                
                print(f"Gap {i+1}: {gap_duration:.2f}s between {gap_start_time:.2f}s and {gap_end_time:.2f}s")
        
        # Calculate frequency over time (in sliding windows)
        window_size = 10  # seconds
        step_size = 5     # seconds
        
        frequencies = []
        window_centers = []
        
        current_window = time_min
        while current_window < time_max - window_size:
            window_end = current_window + window_size
            window_data = location_df[(location_df['seconds_elapsed'] >= current_window) & 
                                     (location_df['seconds_elapsed'] < window_end)]
            
            if len(window_data) > 1:
                window_frequency = len(window_data) / window_size
                frequencies.append(window_frequency)
                window_centers.append(current_window + window_size/2)
            
            current_window += step_size
        
        # Group intervals into bins to see the distribution
        # Define sensible bins based on the data
        max_bin = min(10, max_interval)  # Cap at 10 seconds for readability
        bins = np.linspace(0, max_bin, 50)
        
        # Create visualizations
        plt.figure(figsize=(15, 10))
        
        # Plot 1: Histogram of intervals
        plt.subplot(2, 2, 1)
        plt.hist(intervals, bins=bins, alpha=0.7)
        plt.title('Distribution of Intervals Between Location Records')
        plt.xlabel('Interval (seconds)')
        plt.ylabel('Frequency')
        plt.axvline(median_interval, color='r', linestyle='--', label=f'Median: {median_interval:.4f}s')
        plt.axvline(mean_interval, color='g', linestyle='-.', label=f'Mean: {mean_interval:.4f}s')
        plt.legend()
        plt.grid(alpha=0.3)
        
        # Plot 2: Intervals over time
        plt.subplot(2, 2, 2)
        plt.plot(location_df['seconds_elapsed'].iloc[:-1], intervals, 'b-', alpha=0.5)
        plt.axhline(median_interval, color='r', linestyle='--', label=f'Median: {median_interval:.4f}s')
        plt.title('Intervals Between Records Over Time')
        plt.xlabel('Time (seconds)')
        plt.ylabel('Interval to Next Record (seconds)')
        plt.ylim(0, min(max_interval * 1.1, 5))  # Cap y-axis for readability
        plt.legend()
        plt.grid(alpha=0.3)
        
        # Plot 3: Frequency over time (sliding window)
        plt.subplot(2, 2, 3)
        plt.plot(window_centers, frequencies, 'g-')
        plt.axhline(overall_frequency, color='r', linestyle='--', 
                   label=f'Overall avg: {overall_frequency:.4f} Hz')
        plt.title(f'Location Frequency Over Time ({window_size}s windows)')
        plt.xlabel('Time (seconds)')
        plt.ylabel('Frequency (Hz)')
        plt.legend()
        plt.grid(alpha=0.3)
        
        # Plot 4: Location points colored by interval to next point
        plt.subplot(2, 2, 4)
        scatter = plt.scatter(location_df['longitude'].iloc[:-1], 
                             location_df['latitude'].iloc[:-1],
                             c=intervals, 
                             cmap='viridis', 
                             alpha=0.7,
                             s=10)
        plt.colorbar(scatter, label='Interval to next point (seconds)')
        plt.title('GPS Points Colored by Interval to Next Point')
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.axis('equal')
        plt.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Additional analysis: Check if frequency correlates with GPS accuracy
        if 'horizontalAccuracy' in location_df.columns:
            # Calculate correlation
            accuracy_intervals = pd.DataFrame({
                'horizontalAccuracy': location_df['horizontalAccuracy'].iloc[:-1].values,
                'interval': intervals
            })
            correlation = accuracy_intervals.corr().iloc[0, 1]
            
            print(f"\nCorrelation between GPS accuracy and interval: {correlation:.4f}")
            
            # Visualize relationship between accuracy and intervals
            plt.figure(figsize=(10, 6))
            plt.scatter(accuracy_intervals['horizontalAccuracy'], 
                       accuracy_intervals['interval'],
                       alpha=0.5)
            plt.title('Relationship Between GPS Accuracy and Sampling Interval')
            plt.xlabel('Horizontal Accuracy (meters)')
            plt.ylabel('Interval to Next Point (seconds)')
            plt.grid(alpha=0.3)
            plt.show()
            
            # Group by accuracy ranges
            accuracy_bins = [0, 5, 10, 20, 50, 100, float('inf')]
            accuracy_labels = ['<5m', '5-10m', '10-20m', '20-50m', '50-100m', '>100m']
            
            accuracy_intervals['accuracy_group'] = pd.cut(
                accuracy_intervals['horizontalAccuracy'], 
                bins=accuracy_bins, 
                labels=accuracy_labels
            )
            
            grouped = accuracy_intervals.groupby('accuracy_group')['interval'].agg(['mean', 'median', 'count'])
            grouped['frequency'] = 1 / grouped['median']
            
            print("\nFrequency by GPS accuracy range:")
            print(grouped)
            
            # Plot frequency by accuracy group
            plt.figure(figsize=(10, 6))
            plt.bar(grouped.index, grouped['frequency'])
            plt.title('GPS Update Frequency by Accuracy Range')
            plt.xlabel('GPS Accuracy')
            plt.ylabel('Frequency (Hz)')
            plt.grid(axis='y', alpha=0.3)
            plt.show()
            
except Exception as e:
    print(f"Error analyzing Location data frequency: {e}")
    import traceback
    traceback.print_exc()
finally:
    conn.close()

In [ ]:
def integrate_location_and_quaternion(location_df, quaternion_df):
    # 1. Ensure data is sorted by time
    location_df = location_df.sort_values('seconds_elapsed').reset_index(drop=True)
    quaternion_df = quaternion_df.sort_values('seconds_elapsed').reset_index(drop=True)
    
    # 2. Find common time range
    common_start = max(location_df['seconds_elapsed'].min(), quaternion_df['seconds_elapsed'].min())
    common_end = min(location_df['seconds_elapsed'].max(), quaternion_df['seconds_elapsed'].max())
    
    # 3. Filter data to common time range
    location_df = location_df[(location_df['seconds_elapsed'] >= common_start) & 
                             (location_df['seconds_elapsed'] <= common_end)]
    quaternion_df = quaternion_df[(quaternion_df['seconds_elapsed'] >= common_start) & 
                                 (quaternion_df['seconds_elapsed'] <= common_end)]
    
    # 4. Create interpolation functions for location data
    interp_funcs = {}
    for col in location_df.columns:
        if col != 'seconds_elapsed' and col != 'time':
            try:
                interp_funcs[col] = interp1d(
                    location_df['seconds_elapsed'], 
                    location_df[col], 
                    kind='linear', 
                    bounds_error=False,
                    fill_value=(location_df[col].iloc[0], location_df[col].iloc[-1])
                )
            except Exception as e:
                print(f"Could not create interpolation for {col}: {e}")
    
    # 5. Apply interpolation to quaternion timestamps
    combined_data = []
    for _, row in quaternion_df.iterrows():
        new_row = {'seconds_elapsed': row['seconds_elapsed']}
        
        # Add quaternion data
        for col in quaternion_df.columns:
            if col != 'seconds_elapsed':
                new_row[col] = row[col]
        
        # Add interpolated location data
        for col, interp_func in interp_funcs.items():
            new_row[col] = float(interp_func(row['seconds_elapsed']))
        
        combined_data.append(new_row)
    
    # 6. Create combined DataFrame
    combined_df = pd.DataFrame(combined_data)
    
    # 7. Add derived metrics
    combined_df = enrich_data(combined_df)
    
    return combined_df

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter, butter, filtfilt
import quaternion  # Requires 'numpy-quaternion' package
from datetime import datetime

# Connect to the database
db_path = "./b.sqlite"
conn = sqlite3.connect(db_path)

def create_smoothed_kepler_data():
    try:
        # 1. Load Location data
        location_df = pd.read_sql_query("""
            SELECT time, seconds_elapsed, latitude, longitude, altitude, 
                   speed, horizontalAccuracy, verticalAccuracy, bearing
            FROM Location
            ORDER BY seconds_elapsed
        """, conn)
        
        # 2. Load Orientation/Quaternion data
        quaternion_df = pd.read_sql_query("""
            SELECT time, seconds_elapsed, qx, qy, qz, qw, yaw, pitch, roll
            FROM Orientation
            ORDER BY seconds_elapsed
        """, conn)
        
        print(f"Location data points: {len(location_df)}")
        print(f"Quaternion data points: {len(quaternion_df)}")
        
        # 3. Smooth the quaternion data using different techniques
        smoothed_quaternion_df = smooth_quaternion_data(quaternion_df)
        
        # 4. Find common time range
        common_start = max(location_df['seconds_elapsed'].min(), smoothed_quaternion_df['seconds_elapsed'].min())
        common_end = min(location_df['seconds_elapsed'].max(), smoothed_quaternion_df['seconds_elapsed'].max())
        
        print(f"Common time range: {common_start:.2f}s to {common_end:.2f}s")
        
        # 5. Filter data to common time range
        location_df = location_df[(location_df['seconds_elapsed'] >= common_start) & 
                                 (location_df['seconds_elapsed'] <= common_end)]
        smoothed_quaternion_df = smoothed_quaternion_df[
            (smoothed_quaternion_df['seconds_elapsed'] >= common_start) & 
            (smoothed_quaternion_df['seconds_elapsed'] <= common_end)
        ]
        
        # 6. Apply smoothing to the GPS data as well
        smoothed_location_df = smooth_gps_data(location_df)
        
        # 7. Create interpolation functions for all smoothed quaternion data
        interp_funcs = {}
        for col in smoothed_quaternion_df.columns:
            if col not in ['time', 'seconds_elapsed']:
                try:
                    interp_funcs[col] = interp1d(
                        smoothed_quaternion_df['seconds_elapsed'], 
                        smoothed_quaternion_df[col], 
                        kind='linear', 
                        bounds_error=False,
                        fill_value=(smoothed_quaternion_df[col].iloc[0], smoothed_quaternion_df[col].iloc[-1])
                    )
                except Exception as e:
                    print(f"Could not create interpolation for {col}: {e}")
        
        # 8. Add interpolated quaternion data to location dataframe
        for col, interp_func in interp_funcs.items():
            smoothed_location_df[col] = smoothed_location_df['seconds_elapsed'].apply(lambda x: float(interp_func(x)))
        
        # 9. Add timestamp in ISO format for Kepler
        base_time = pd.to_datetime('today')
        smoothed_location_df['timestamp'] = base_time + pd.to_timedelta(
            smoothed_location_df['seconds_elapsed'] - common_start, unit='s')
        smoothed_location_df['timestamp_str'] = smoothed_location_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
        
        # 10. Calculate useful derived metrics
        add_derived_metrics(smoothed_location_df)
        
        # 11. Create a version with both raw and smoothed data for comparison
        comparison_df = create_comparison_dataset(location_df, smoothed_location_df, smoothed_quaternion_df, interp_funcs)
        
        # 12. Save outputs to CSV
        smoothed_location_df.to_csv('kepler_smoothed_data.csv', index=False)
        comparison_df.to_csv('kepler_raw_vs_smoothed.csv', index=False)
        
        print(f"Smoothed data saved to kepler_smoothed_data.csv")
        print(f"Comparison data saved to kepler_raw_vs_smoothed.csv")
        print("\nColumns available in the smoothed data:")
        print("\n".join(f"- {col}" for col in smoothed_location_df.columns))
        
        print("\nKepler.gl Configuration Tips:")
        print("1. Use timestamp_str as the Time column for animation")
        print("2. To compare raw vs smoothed data in the comparison CSV:")
        print("   - Create two layers with the same styling but different columns")
        print("   - Use raw_latitude/raw_longitude for one layer")
        print("   - Use smoothed_latitude/smoothed_longitude for the other layer")
        print("   - Try different coloring by quaternion metrics like qx_sgolay, qx_butter, etc.")
        
    except Exception as e:
        print(f"Error creating smoothed data: {e}")
        import traceback
        traceback.print_exc()
    finally:
        conn.close()

def smooth_quaternion_data(quat_df):
    """Apply various smoothing techniques to quaternion data."""
    smoothed_df = quat_df.copy()
    
    # 1. Savitzky-Golay filter (good for preserving peaks)
    for col in ['qx', 'qy', 'qz', 'qw']:
        if col in smoothed_df.columns:
            # Parameters: window length must be odd, polynomial order
            window_length = min(21, len(smoothed_df) // 10 * 2 + 1)  # Adaptive window size
            smoothed_df[f'{col}_sgolay'] = savgol_filter(
                smoothed_df[col], window_length=window_length, polyorder=3)
    
    # 2. Butterworth low-pass filter (good for removing high-frequency noise)
    def butter_lowpass_filter(data, cutoff, fs, order=5):
        nyquist = 0.5 * fs
        normal_cutoff = cutoff / nyquist
        b, a = butter(order, normal_cutoff, btype='low', analog=False)
        return filtfilt(b, a, data)
    
    # Estimate sampling rate
    avg_interval = np.diff(smoothed_df['seconds_elapsed']).mean()
    sampling_rate = 1.0 / avg_interval if avg_interval > 0 else 100  # Hz
    
    for col in ['qx', 'qy', 'qz', 'qw']:
        if col in smoothed_df.columns:
            # Cutoff frequency as fraction of sampling rate
            smoothed_df[f'{col}_butter'] = butter_lowpass_filter(
                smoothed_df[col], cutoff=2.0, fs=sampling_rate, order=4)
    
    # 3. Moving average (simplest approach)
    window_size = max(5, len(smoothed_df) // 100)  # Adaptive window size
    for col in ['qx', 'qy', 'qz', 'qw']:
        if col in smoothed_df.columns:
            smoothed_df[f'{col}_mavg'] = smoothed_df[col].rolling(
                window=window_size, center=True, min_periods=1).mean()
    
    # 4. Proper quaternion SLERP smoothing (if numpy-quaternion package is available)
    try:
        if all(col in smoothed_df.columns for col in ['qw', 'qx', 'qy', 'qz']):
            # Convert to numpy quaternions
            quats = np.quaternion.from_float_array(
                smoothed_df[['qw', 'qx', 'qy', 'qz']].values)
            
            # Apply quaternion-specific smoothing
            smoothed_quats = smooth_quaternions(quats, window_size=window_size)
            
            # Convert back to components
            quat_array = np.quaternion.as_float_array(smoothed_quats)
            smoothed_df['qw_slerp'] = quat_array[:, 0]
            smoothed_df['qx_slerp'] = quat_array[:, 1]
            smoothed_df['qy_slerp'] = quat_array[:, 2]
            smoothed_df['qz_slerp'] = quat_array[:, 3]
    except Exception as e:
        print(f"Could not perform quaternion SLERP smoothing: {e}")
    
    # 5. Apply smoothing to Euler angles (roll, pitch, yaw) if available
    for col in ['roll', 'pitch', 'yaw']:
        if col in smoothed_df.columns:
            # Savitzky-Golay for Euler angles
            window_length = min(21, len(smoothed_df) // 10 * 2 + 1)
            smoothed_df[f'{col}_smooth'] = savgol_filter(
                smoothed_df[col], window_length=window_length, polyorder=3)
    
    return smoothed_df

def smooth_quaternions(quats, window_size=5):
    """Apply proper quaternion smoothing using SLERP."""
    smoothed = np.copy(quats)
    half_window = window_size // 2
    
    for i in range(len(quats)):
        start_idx = max(0, i - half_window)
        end_idx = min(len(quats), i + half_window + 1)
        
        # Get quaternions in the window
        window_quats = quats[start_idx:end_idx]
        
        # Simple approach: average the quaternions
        # Note: This is a simplification - proper quaternion averaging would use
        # techniques like quaternion filtering or averaging on a manifold
        avg_quat = np.mean(window_quats)
        smoothed[i] = avg_quat / np.abs(avg_quat)  # Normalize
    
    return smoothed

def smooth_gps_data(location_df):
    """Apply appropriate smoothing to GPS data."""
    smoothed_df = location_df.copy()
    
    # Check if we have enough points for smoothing
    if len(smoothed_df) < 5:
        print("Not enough GPS points for smoothing, returning original data")
        return smoothed_df
    
    # 1. Moving average for simple smoothing
    window_size = max(3, len(smoothed_df) // 50)  # Adaptive window size
    
    for col in ['latitude', 'longitude', 'altitude']:
        if col in smoothed_df.columns:
            smoothed_df[f'{col}_raw'] = smoothed_df[col].copy()  # Keep original
            smoothed_df[col] = smoothed_df[col].rolling(
                window=window_size, center=True, min_periods=1).mean()
    
    # 2. Savitzky-Golay filter for smoother curves while preserving shape
    for col in ['speed', 'bearing']:
        if col in smoothed_df.columns:
            window_length = min(11, len(smoothed_df) // 5 * 2 + 1)  # Must be odd
            if window_length > 3:  # Minimum required window size
                smoothed_df[f'{col}_raw'] = smoothed_df[col].copy()  # Keep original
                smoothed_df[col] = savgol_filter(
                    smoothed_df[col], window_length=window_length, polyorder=2)
    
    # 3. Kalman filter for GPS path (if we have enough points)
    try:
        if len(smoothed_df) >= 10 and all(col in smoothed_df.columns for col in ['latitude', 'longitude']):
            from filterpy.kalman import KalmanFilter
            
            # Setup Kalman filter
            kf = KalmanFilter(dim_x=4, dim_z=2)  # [lat, lon, lat_vel, lon_vel]
            
            # State transition matrix (constant velocity model)
            dt = np.diff(smoothed_df['seconds_elapsed']).mean()  # Average time step
            kf.F = np.array([
                [1, 0, dt, 0],
                [0, 1, 0, dt],
                [0, 0, 1, 0],
                [0, 0, 0, 1]
            ])
            
            # Measurement function (we only measure position)
            kf.H = np.array([
                [1, 0, 0, 0],
                [0, 1, 0, 0]
            ])
            
            # Measurement noise (adjust based on GPS accuracy)
            if 'horizontalAccuracy' in smoothed_df.columns:
                avg_accuracy = smoothed_df['horizontalAccuracy'].mean()
                r_scale = max(1e-6, min(1e-4, avg_accuracy * 1e-6))  # Scale based on accuracy
            else:
                r_scale = 1e-5  # Default value
                
            kf.R = np.array([[r_scale, 0],
                             [0, r_scale]])
            
            # Process noise
            q_scale = 1e-6  # Adjust as needed
            kf.Q = np.eye(4) * q_scale
            
            # Initial state
            kf.x = np.array([
                smoothed_df['latitude'].iloc[0],
                smoothed_df['longitude'].iloc[0],
                0,  # Initial velocity
                0   # Initial velocity
            ])
            
            # Initial state covariance
            kf.P = np.eye(4) * 1.0
            
            # Apply Kalman filter
            kalman_lat = []
            kalman_lon = []
            
            for i in range(len(smoothed_df)):
                # Predict
                kf.predict()
                
                # Update with measurement
                measurement = np.array([
                    smoothed_df['latitude'].iloc[i],
                    smoothed_df['longitude'].iloc[i]
                ])
                kf.update(measurement)
                
                # Store filtered values
                kalman_lat.append(kf.x[0])
                kalman_lon.append(kf.x[1])
            
            smoothed_df['latitude_kalman'] = kalman_lat
            smoothed_df['longitude_kalman'] = kalman_lon
            
            # Use Kalman filtered values as primary
            smoothed_df['latitude'] = kalman_lat
            smoothed_df['longitude'] = kalman_lon
            
    except Exception as e:
        print(f"Could not apply Kalman filter: {e}")
    
    return smoothed_df

def add_derived_metrics(df):
    """Add derived metrics useful for visualization."""
    
    # 1. Calculate speed in km/h if not already present
    if 'speed' in df.columns and 'speed_kmh' not in df.columns:
        df['speed_kmh'] = df['speed'] * 3.6
    
    # 2. Normalize quaternion components for easier coloring
    for col_prefix in ['qx', 'qy', 'qz', 'qw']:
        # Normalize each smoothed version
        for suffix in ['', '_sgolay', '_butter', '_mavg', '_slerp']:
            col = f"{col_prefix}{suffix}"
            if col in df.columns:
                min_val = df[col].min()
                max_val = df[col].max()
                if max_val > min_val:  # Avoid division by zero
                    df[f"{col}_norm"] = (df[col] - min_val) / (max_val - min_val)
    
    # 3. Calculate turn rate from yaw or quaternions
    if 'yaw_smooth' in df.columns:
        # Use smoothed yaw for better turn rate calculation
        df['turn_rate'] = df['yaw_smooth'].diff() / df['seconds_elapsed'].diff()
    elif 'yaw' in df.columns:
        df['turn_rate'] = df['yaw'].diff() / df['seconds_elapsed'].diff()
    
    # Normalize turn rate for coloring
    if 'turn_rate' in df.columns:
        # Use percentiles to avoid outliers affecting the normalization
        p05 = df['turn_rate'].quantile(0.05)
        p95 = df['turn_rate'].quantile(0.95)
        # Normalize and clip to avoid extreme values
        df['turn_rate_norm'] = ((df['turn_rate'] - p05) / (p95 - p05)).clip(0, 1)
    
    # 4. Calculate acceleration
    if 'speed' in df.columns:
        df['acceleration'] = df['speed'].diff() / df['seconds_elapsed'].diff()
        # Normalize acceleration
        if 'acceleration' in df.columns:
            p05 = df['acceleration'].quantile(0.05)
            p95 = df['acceleration'].quantile(0.95)
            df['acceleration_norm'] = ((df['acceleration'] - p05) / (p95 - p05)).clip(0, 1)
    
    # 5. Detect maneuvers (combining turn rate and speed)
    if 'turn_rate_norm' in df.columns and 'speed_kmh' in df.columns:
        # High turn_rate + high speed = exciting maneuver
        max_speed = df['speed_kmh'].max()
        speed_norm = df['speed_kmh'] / max_speed if max_speed > 0 else 0
        df['maneuver_intensity'] = df['turn_rate_norm'] * speed_norm
    
    # 6. Calculate heading change rate for each smoothed quaternion
    for suffix in ['', '_sgolay', '_butter', '_mavg', '_slerp']:
        qw_col = f'qw{suffix}'
        qx_col = f'qx{suffix}'
        qy_col = f'qy{suffix}'
        qz_col = f'qz{suffix}'
        
        if all(col in df.columns for col in [qw_col, qx_col, qy_col, qz_col]):
            # Calculate heading change rate from quaternion components
            # This is a simplified calculation - proper quaternion diff would be better
            df[f'quat_rate{suffix}'] = np.sqrt(
                df[qx_col].diff()**2 + 
                df[qy_col].diff()**2 + 
                df[qz_col].diff()**2 + 
                df[qw_col].diff()**2
            ) / df['seconds_elapsed'].diff()
            
            # Normalize for visualization
            rate_col = f'quat_rate{suffix}'
            if rate_col in df.columns:
                # Use percentiles for better normalization
                p95 = df[rate_col].quantile(0.95)
                if p95 > 0:
                    df[f'{rate_col}_norm'] = (df[rate_col] / p95).clip(0, 1)
    
    # 7. Create composite metrics for better visualization
    # Combine speed with orientation change
    if 'speed_kmh' in df.columns and 'quat_rate_norm' in df.columns:
        speed_norm = df['speed_kmh'] / df['speed_kmh'].max() if df['speed_kmh'].max() > 0 else 0
        df['speed_turn_composite'] = (speed_norm * 0.5 + df['quat_rate_norm'] * 0.5)
    
    # 8. Distance traveled (cumulative)
    if all(col in df.columns for col in ['latitude', 'longitude']):
        from geopy.distance import geodesic
        
        distances = [0]  # First point has no distance
        
        for i in range(1, len(df)):
            point1 = (df['latitude'].iloc[i-1], df['longitude'].iloc[i-1])
            point2 = (df['latitude'].iloc[i], df['longitude'].iloc[i])
            
            # Calculate distance in meters
            try:
                distance = geodesic(point1, point2).meters
                # Filter out unrealistic jumps (likely GPS errors)
                if distance > 100 and 'speed' in df.columns:
                    # Check if distance is reasonable given the speed and time
                    time_diff = df['seconds_elapsed'].iloc[i] - df['seconds_elapsed'].iloc[i-1]
                    speed = df['speed'].iloc[i-1]
                    max_reasonable_dist = speed * time_diff * 2  # Allow some buffer
                    
                    if distance > max_reasonable_dist:
                        distance = max_reasonable_dist
            except:
                distance = 0
                
            distances.append(distances[-1] + distance)
        
        df['distance_m'] = distances
        df['distance_km'] = np.array(distances) / 1000
    
    return df

def create_comparison_dataset(raw_loc_df, smoothed_loc_df, smoothed_quat_df, interp_funcs):
    """Create a dataset that includes both raw and smoothed data for comparison."""
    
    # Start with the raw location data
    comparison_df = raw_loc_df.copy()
    
    # Keep original lat/lon as raw_latitude and raw_longitude
    comparison_df['raw_latitude'] = comparison_df['latitude']
    comparison_df['raw_longitude'] = comparison_df['longitude']
    
    # Add smoothed lat/lon
    comparison_df['smoothed_latitude'] = smoothed_loc_df['latitude']
    comparison_df['smoothed_longitude'] = smoothed_loc_df['longitude']
    
    # If we have Kalman-filtered coordinates, include those too
    if 'latitude_kalman' in smoothed_loc_df.columns:
        comparison_df['kalman_latitude'] = smoothed_loc_df['latitude_kalman']
        comparison_df['kalman_longitude'] = smoothed_loc_df['longitude_kalman']
    
    # Add interpolated quaternion data (both raw and smoothed versions)
    for col, interp_func in interp_funcs.items():
        comparison_df[col] = comparison_df['seconds_elapsed'].apply(lambda x: float(interp_func(x)))
    
    # Add timestamp for Kepler
    base_time = pd.to_datetime('today')
    comparison_df['timestamp'] = base_time + pd.to_timedelta(
        comparison_df['seconds_elapsed'] - comparison_df['seconds_elapsed'].min(), unit='s')
    comparison_df['timestamp_str'] = comparison_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    
    # Add useful derived metrics
    add_derived_metrics(comparison_df)
    
    return comparison_df

# Execute the function
create_smoothed_kepler_data()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Sample data - you'll need to replace this with loading your actual data
# For demonstration purposes, I'm creating sample data that mimics your graph
def load_data():
    # Replace this with your actual data loading code
    # Example: data = pd.read_csv('your_data.csv')
    # Return time and qx columns
    
    # For demo purposes only:
    time = np.arange(0, 800, 0.5)
    qx = np.sin(time * 0.05) * 0.9  # Creates a wave pattern similar to your data
    return time, qx

# Detect laps based on QX crossings
def detect_laps(time, qx, threshold=0.7):
    # Find where QX crosses from negative to positive with steep slope
    # This seems to indicate the start of a lap based on your plot
    
    # Get indices where qx crosses from negative to positive through threshold
    lap_starts = []
    for i in range(1, len(qx)):
        if qx[i-1] < threshold and qx[i] >= threshold:
            lap_starts.append(i)
    
    # Convert to time values
    lap_start_times = [time[i] for i in lap_starts]
    return lap_start_times

# Plot the data with lap sections highlighted
def plot_laps(time, qx, lap_start_times):
    plt.figure(figsize=(12, 6))
    
    # Plot raw QX values
    plt.plot(time, qx, 'b-', label='QX Values')
    
    # Highlight lap sections
    for i, start_time in enumerate(lap_start_times):
        if i < len(lap_start_times) - 1:
            end_time = lap_start_times[i + 1]
            plt.axvspan(start_time, end_time, alpha=0.2, color=f'C{i%9+1}', 
                       label=f'Lap {i+1}' if i==0 else "")
    
    # Add lap number annotations
    for i, start_time in enumerate(lap_start_times):
        plt.text(start_time, 0.8, f'Lap {i+1}', fontsize=10)
        
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    plt.title('QX Values with Lap Detection')
    plt.xlabel('Time (seconds)')
    plt.ylabel('QX Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Main function
def main():
    time, qx = load_data()
    
    # Use a different approach for more robust lap detection
    # Look for transitions from strongly negative to strongly positive values
    lap_start_times = detect_laps(time, qx)
    
    print(f"Detected {len(lap_start_times)} laps at times: {lap_start_times}")
    
    # Plot the results
    plot_laps(time, qx, lap_start_times)
    
    # Optional: Export lap timing data
    # np.savetxt('lap_times.csv', np.array(lap_start_times), delimiter=',', header='lap_start_time')

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks

# Load the data
def load_data(file_path):
    """
    Load data from CSV or other format
    Expects columns for time and qw values
    """
    try:
        # Try to load as CSV first
        data = pd.read_csv(file_path)
        
        # Extract time and qw columns - adjust column names as needed
        if 'time' in data.columns and 'qw' in data.columns:
            time = data['time'].values
            qw = data['qw'].values
        else:
            # Try to guess columns (first column as time, look for qw in columns)
            qw_cols = [col for col in data.columns if 'qw' in col.lower()]
            if qw_cols:
                time = data.iloc[:, 0].values
                qw = data[qw_cols[0]].values
            else:
                # Fallback to first two columns
                time = data.iloc[:, 0].values
                qw = data.iloc[:, 1].values
        
        print(f"Loaded data with {len(time)} points")
        return time, qw
    except Exception as e:
        print(f"Error loading data: {e}")
        # Generate sample data if file loading fails
        print("Generating sample data instead")
        return generate_sample_data()

def generate_sample_data():
    """Generate sample data similar to the image for testing"""
    time = np.arange(100, 800, 0.5)
    period = 120  # Approximately the period shown in the image
    
    # Base sine wave
    qw = np.sin(2 * np.pi * (time - 100) / period) * 0.9
    
    # Add some noise and irregularities to make it look more like the real data
    noise = np.random.normal(0, 0.1, len(time))
    qw += noise
    
    # Clip to the range [-1, 1] as shown in the image
    qw = np.clip(qw, -1, 1)
    
    return time, qw

# Detect laps based on QW transitions
def detect_laps(time, qw, threshold=0.5):
    """
    Detect laps based on QW value transitions
    Using threshold crossing from negative to positive values
    """
    lap_starts = []
    for i in range(1, len(qw)):
        if qw[i-1] < threshold and qw[i] >= threshold:
            lap_starts.append(i)
    
    # Filter out laps that are too close together (likely noise)
    if len(lap_starts) > 1:
        filtered_starts = [lap_starts[0]]
        min_gap = 30  # Minimum points between lap starts
        
        for i in range(1, len(lap_starts)):
            if lap_starts[i] - filtered_starts[-1] > min_gap:
                filtered_starts.append(lap_starts[i])
        
        lap_starts = filtered_starts
    
    # Convert indices to time values
    lap_start_times = [time[i] for i in lap_starts]
    return lap_start_times

# Plot the data with lap sections highlighted
def plot_laps(time, qw, lap_start_times):
    plt.figure(figsize=(14, 7))
    
    # Plot raw QW values
    plt.plot(time, qw, 'b-', linewidth=1, label='Raw qw values')
    
    # Compute and plot a moving average to smooth the data
    window = 51
    if len(qw) > window:
        qw_smoothed = np.convolve(qw, np.ones(window)/window, mode='valid')
        time_smoothed = time[window//2:-(window//2)]
        plt.plot(time_smoothed, qw_smoothed, 'k-', linewidth=1.5, label=f'{window}-point rolling average')
    
    # Highlight lap sections
    colors = ['#ffcccc', '#ccffcc', '#ccccff', '#ffffcc', '#ffccff', '#ccffff']
    for i in range(len(lap_start_times)):
        start_time = lap_start_times[i]
        if i < len(lap_start_times) - 1:
            end_time = lap_start_times[i + 1]
        else:
            end_time = time[-1]  # Use the last time point for the final lap
            
        color = colors[i % len(colors)]
        plt.axvspan(start_time, end_time, alpha=0.2, color=color, 
                    label=f'Lap {i+1}' if i==0 else "")
    
    # Add lap number annotations
    for i, start_time in enumerate(lap_start_times):
        plt.text(start_time, 0.8, f'Lap {i+1}', fontsize=10, fontweight='bold')
        plt.axvline(x=start_time, color='red', linestyle='--', alpha=0.6)
    
    # Add horizontal line at y=0
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Add title and labels
    plt.title('QW Values Over Time with Lap Detection', fontsize=14)
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('QW Value', fontsize=12)
    plt.ylim(-1.1, 1.1)  # Set y-axis limits as shown in the image
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    
    # Add annotation about QW
    plt.figtext(0.5, 0.01, 'Note: qw is a quaternion component used to represent 3D orientation', 
                ha='center', fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.show()

# Main function
def main():
    # Replace with your actual file path
    file_path = 'your_data_file.csv'
    
    # Try to load real data, fall back to sample data if needed
    try:
        time, qw = load_data(file_path)
    except:
        print("Using sample data")
        time, qw = generate_sample_data()
    
    # Detect laps
    threshold = 0.5  # Adjust this threshold based on your data
    lap_start_times = detect_laps(time, qw, threshold)
    
    print(f"Detected {len(lap_start_times)} laps")
    if lap_start_times:
        print("Lap start times:", lap_start_times)
        
        # Calculate lap durations
        lap_durations = []
        for i in range(len(lap_start_times)-1):
            duration = lap_start_times[i+1] - lap_start_times[i]
            lap_durations.append(duration)
        if lap_durations:
            print("Lap durations (seconds):", [round(d, 1) for d in lap_durations])
            print(f"Average lap time: {round(sum(lap_durations)/len(lap_durations), 2)} seconds")
    
    # Plot the results
    plot_laps(time, qw, lap_start_times)
    
    # Optional: Export lap timing data
    # pd.DataFrame({'lap_start_time': lap_start_times}).to_csv('lap_times.csv', index=False)

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def load_quaternion_data(file_path):
    """
    Load quaternion data from a file.
    
    Parameters:
    file_path (str): Path to the data file (CSV or similar)
    
    Returns:
    tuple: (time, qw, qx) arrays
    """
    try:
        # Try to load as CSV first
        data = pd.read_csv(file_path)
        
        # Try to find the relevant columns
        time_col = None
        qw_col = None
        qx_col = None
        
        # Look for time column
        time_candidates = ['time', 'timestamp', 't']
        for col in time_candidates:
            if col in data.columns:
                time_col = col
                break
        
        # If no time column found, use the first column
        if time_col is None:
            time_col = data.columns[0]
            print(f"Using '{time_col}' as time column")
        
        # Look for qw and qx columns
        for col in data.columns:
            if 'qw' in col.lower():
                qw_col = col
            elif 'qx' in col.lower():
                qx_col = col
        
        # Check if we found the columns
        if qw_col is None or qx_col is None:
            print("Couldn't find qw or qx columns. Available columns:", data.columns.tolist())
            # Try some common alternatives or guesses
            if qw_col is None and 'q0' in data.columns:
                qw_col = 'q0'
                print(f"Using '{qw_col}' as qw")
            if qx_col is None and 'q1' in data.columns:
                qx_col = 'q1'
                print(f"Using '{qx_col}' as qx")
        
        # Final check and return the data
        if qw_col is not None and qx_col is not None:
            time = data[time_col].values
            qw = data[qw_col].values
            qx = data[qx_col].values
            print(f"Loaded {len(time)} data points")
            return time, qw, qx
        else:
            raise ValueError("Couldn't identify quaternion columns")
            
    except Exception as e:
        print(f"Error loading data: {e}")
        # Generate sample data as fallback
        return generate_sample_data()

def generate_sample_data():
    """Generate sample quaternion data for testing"""
    print("Generating sample data")
    time = np.arange(100, 800, 0.5)
    period = 120  # Approximately the period shown in the previous image
    
    # Generate slightly different patterns for qw and qx
    qw = np.sin(2 * np.pi * (time - 100) / period) * 0.9
    qx = np.sin(2 * np.pi * (time - 100) / period + np.pi/2) * 0.9  # Phase shifted
    
    # Add some noise to make it look more realistic
    qw += np.random.normal(0, 0.05, len(time))
    qx += np.random.normal(0, 0.05, len(time))
    
    # Clip to the range [-1, 1] as quaternion components should be
    qw = np.clip(qw, -1, 1)
    qx = np.clip(qx, -1, 1)
    
    return time, qw, qx

def plot_quaternion_components(time, qw, qx):
    """
    Plot QW and QX values over time.
    
    Parameters:
    time (array): Time values
    qw (array): QW quaternion component values
    qx (array): QX quaternion component values
    """
    plt.figure(figsize=(12, 8))
    
    # Create two subplots vertically stacked
    ax1 = plt.subplot(211)  # First subplot for QW
    ax2 = plt.subplot(212, sharex=ax1)  # Second subplot for QX, sharing x-axis
    
    # Plot QW values
    ax1.plot(time, qw, 'b-', linewidth=1.2)
    ax1.set_ylabel('QW Value', fontsize=12)
    ax1.set_title('QW Values Over Time', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_ylim(-1.1, 1.1)
    
    # Plot QX values
    ax2.plot(time, qx, 'r-', linewidth=1.2)
    ax2.set_xlabel('Time (seconds)', fontsize=12)
    ax2.set_ylabel('QX Value', fontsize=12)
    ax2.set_title('QX Values Over Time', fontsize=14)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylim(-1.1, 1.1)
    
    # Add a note about quaternions
    plt.figtext(0.5, 0.01, 'Note: qw and qx are quaternion components used to represent 3D orientation', 
                ha='center', fontsize=10, style='italic')
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3)
    plt.show()
    
    # Optionally, also create a single plot with both components
    plt.figure(figsize=(12, 6))
    plt.plot(time, qw, 'b-', linewidth=1.2, label='QW')
    plt.plot(time, qx, 'r-', linewidth=1.2, label='QX')
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('Quaternion Value', fontsize=12)
    plt.title('Quaternion Components Over Time', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    plt.legend()
    plt.ylim(-1.1, 1.1)
    plt.tight_layout()
    plt.show()

def main():
    # Replace with your actual file path
    file_path = 'your_data_file.csv'
    
    # Load data
    try:
        time, qw, qx = load_quaternion_data(file_path)
    except:
        print("Error loading data, using sample data instead")
        time, qw, qx = generate_sample_data()
    
    # Create the plots
    plot_quaternion_components(time, qw, qx)

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import pandas as pd

def load_quaternion_data_from_sqlite(db_path, table_name='Orientation'):
    """
    Load quaternion data from a SQLite database.
    
    Parameters:
    db_path (str): Path to the SQLite database file
    table_name (str): Name of the table containing quaternion data
    
    Returns:
    tuple: (time, qw, qx) arrays
    """
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_path)
        
        # First, check if the table exists and what columns it has
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [row[1] for row in cursor.fetchall()]
        
        print(f"Found columns in {table_name} table: {columns}")
        
        # Look for time/timestamp column
        time_col = None
        for candidate in ['time', 'timestamp', 't', 'Time', 'Timestamp']:
            if candidate in columns:
                time_col = candidate
                break
        
        # If no explicit time column, look for one with 'time' in it
        if time_col is None:
            for col in columns:
                if 'time' in col.lower():
                    time_col = col
                    break
        
        # Look for quaternion component columns
        qw_col = None
        qx_col = None
        
        # Common naming patterns for quaternion components
        for col in columns:
            col_lower = col.lower()
            if 'qw' in col_lower or 'q0' in col_lower or 'q_w' in col_lower:
                qw_col = col
            elif 'qx' in col_lower or 'q1' in col_lower or 'q_x' in col_lower:
                qx_col = col
        
        if not time_col or not qw_col or not qx_col:
            print("Warning: Couldn't automatically identify all required columns.")
            print(f"Time column: {time_col}")
            print(f"QW column: {qw_col}")
            print(f"QX column: {qx_col}")
            
            # Make best guesses if columns weren't found
            if not time_col:
                time_col = columns[0]  # Assume first column is time
                print(f"Using '{time_col}' as time column")
            if not qw_col and len(columns) > 1:
                qw_col = columns[1]  # Assume second column might be qw
                print(f"Using '{qw_col}' as QW column")
            if not qx_col and len(columns) > 2:
                qx_col = columns[2]  # Assume third column might be qx
                print(f"Using '{qx_col}' as QX column")
        
        # Load the data into a pandas DataFrame
        query = f"SELECT {time_col}, {qw_col}, {qx_col} FROM {table_name}"
        df = pd.read_sql_query(query, conn)
        
        # Close the connection
        conn.close()
        
        # Extract arrays
        time = df[time_col].values
        qw = df[qw_col].values
        qx = df[qx_col].values
        
        print(f"Loaded {len(time)} data points from SQLite database")
        return time, qw, qx
        
    except Exception as e:
        print(f"Error loading data from SQLite database: {e}")
        # Generate sample data as fallback
        return generate_sample_data()

def generate_sample_data():
    """Generate sample quaternion data for testing"""
    print("Generating sample data")
    time = np.arange(100, 800, 0.5)
    period = 120  # Approximately the period shown in the previous image
    
    # Generate slightly different patterns for qw and qx
    qw = np.sin(2 * np.pi * (time - 100) / period) * 0.9
    qx = np.sin(2 * np.pi * (time - 100) / period + np.pi/2) * 0.9  # Phase shifted
    
    # Add some noise to make it look more realistic
    qw += np.random.normal(0, 0.05, len(time))
    qx += np.random.normal(0, 0.05, len(time))
    
    # Clip to the range [-1, 1] as quaternion components should be
    qw = np.clip(qw, -1, 1)
    qx = np.clip(qx, -1, 1)
    
    return time, qw, qx

def plot_quaternion_components(time, qw, qx):
    """
    Plot QW and QX values over time.
    
    Parameters:
    time (array): Time values
    qw (array): QW quaternion component values
    qx (array): QX quaternion component values
    """
    plt.figure(figsize=(12, 8))
    
    # Create two subplots vertically stacked
    ax1 = plt.subplot(211)  # First subplot for QW
    ax2 = plt.subplot(212, sharex=ax1)  # Second subplot for QX, sharing x-axis
    
    # Plot QW values
    ax1.plot(time, qw, 'b-', linewidth=1.2)
    ax1.set_ylabel('QW Value', fontsize=12)
    ax1.set_title('QW Values Over Time', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_ylim(-1.1, 1.1)
    
    # Plot QX values
    ax2.plot(time, qx, 'r-', linewidth=1.2)
    ax2.set_xlabel('Time (seconds)', fontsize=12)
    ax2.set_ylabel('QX Value', fontsize=12)
    ax2.set_title('QX Values Over Time', fontsize=14)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylim(-1.1, 1.1)
    
    # Add a note about quaternions
    plt.figtext(0.5, 0.01, 'Note: qw and qx are quaternion components used to represent 3D orientation', 
                ha='center', fontsize=10, style='italic')
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3)
    plt.show()
    
    # Optionally, also create a single plot with both components
    plt.figure(figsize=(12, 6))
    plt.plot(time, qw, 'b-', linewidth=1.2, label='QW')
    plt.plot(time, qx, 'r-', linewidth=1.2, label='QX')
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('Quaternion Value', fontsize=12)
    plt.title('Quaternion Components Over Time', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    plt.legend()
    plt.ylim(-1.1, 1.1)
    plt.tight_layout()
    plt.show()

def main():
    # Specify the path to your SQLite database
    db_path = 'b.sqlite'
    
    # Load data from the Orientation table in the SQLite database
    time, qw, qx = load_quaternion_data_from_sqlite(db_path, table_name='Orientation')
    
    # Create the plots
    plot_quaternion_components(time, qw, qx)

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sqlite3
from scipy.signal import find_peaks

# Load data from SQLite database
def load_data_from_sqlite(db_path='b.sqlite', table_name='Orientation'):
    """
    Load data from SQLite database
    Expects table with time and qw values
    """
    try:
        # Connect to SQLite database
        conn = sqlite3.connect(db_path)
        print(f"Connected to database: {db_path}")
        
        # First, check what columns are available
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [row[1] for row in cursor.fetchall()]
        print(f"Available columns: {columns}")
        
        # Find time column
        time_col = None
        for col in columns:
            if col.lower() in ['time', 'timestamp', 't']:
                time_col = col
                break
        
        # If no specific time column found, use first column or look for one with 'time' in it
        if not time_col:
            for col in columns:
                if 'time' in col.lower():
                    time_col = col
                    break
            if not time_col:
                time_col = columns[0]
        
        # Find qw column
        qw_col = None
        for col in columns:
            if col.lower() in ['qw', 'q0', 'q_w']:
                qw_col = col
                break
        
        # If no specific qw column found, look for one with 'qw' in it
        if not qw_col:
            for col in columns:
                if 'qw' in col.lower():
                    qw_col = col
                    break
        
        if not qw_col:
            raise ValueError(f"Could not find QW column in {columns}")
        
        print(f"Using columns: time={time_col}, qw={qw_col}")
        
        # Query data
        query = f"SELECT {time_col}, {qw_col} FROM {table_name}"
        df = pd.read_sql_query(query, conn)
        
        # Close connection
        conn.close()
        
        # Extract arrays
        time = df[time_col].values
        qw = df[qw_col].values
        
        print(f"Loaded {len(time)} data points from SQLite database")
        return time, qw
        
    except Exception as e:
        print(f"Error loading data from SQLite: {e}")
        # Generate sample data if database loading fails
        print("Generating sample data instead")
        return generate_sample_data()

def generate_sample_data():
    """Generate sample data similar to the image for testing"""
    time = np.arange(100, 800, 0.5)
    period = 120  # Approximately the period shown in the image
    
    # Base sine wave
    qw = np.sin(2 * np.pi * (time - 100) / period) * 0.9
    
    # Add some noise and irregularities to make it look more like the real data
    noise = np.random.normal(0, 0.1, len(time))
    qw += noise
    
    # Clip to the range [-1, 1] as shown in the image
    qw = np.clip(qw, -1, 1)
    
    return time, qw

# Detect laps based on QW transitions
def detect_laps(time, qw, threshold=0.5):
    """
    Detect laps based on QW value transitions
    Using threshold crossing from negative to positive values
    """
    lap_starts = []
    for i in range(1, len(qw)):
        if qw[i-1] < threshold and qw[i] >= threshold:
            lap_starts.append(i)
    
    # Filter out laps that are too close together (likely noise)
    if len(lap_starts) > 1:
        filtered_starts = [lap_starts[0]]
        min_gap = 30  # Minimum points between lap starts
        
        for i in range(1, len(lap_starts)):
            if lap_starts[i] - filtered_starts[-1] > min_gap:
                filtered_starts.append(lap_starts[i])
        
        lap_starts = filtered_starts
    
    # Convert indices to time values
    lap_start_times = [time[i] for i in lap_starts]
    return lap_start_times

# Plot the data with lap sections highlighted
def plot_laps(time, qw, lap_start_times):
    plt.figure(figsize=(14, 7))
    
    # Plot raw QW values
    plt.plot(time, qw, 'b-', linewidth=1, label='Raw qw values')
    
    # Compute and plot a moving average to smooth the data
    window = 51
    if len(qw) > window:
        qw_smoothed = np.convolve(qw, np.ones(window)/window, mode='valid')
        time_smoothed = time[window//2:-(window//2)]
        plt.plot(time_smoothed, qw_smoothed, 'k-', linewidth=1.5, label=f'{window}-point rolling average')
    
    # Highlight lap sections
    colors = ['#ffcccc', '#ccffcc', '#ccccff', '#ffffcc', '#ffccff', '#ccffff']
    for i in range(len(lap_start_times)):
        start_time = lap_start_times[i]
        if i < len(lap_start_times) - 1:
            end_time = lap_start_times[i + 1]
        else:
            end_time = time[-1]  # Use the last time point for the final lap
            
        color = colors[i % len(colors)]
        plt.axvspan(start_time, end_time, alpha=0.2, color=color, 
                    label=f'Lap {i+1}' if i==0 else "")
    
    # Add lap number annotations
    for i, start_time in enumerate(lap_start_times):
        plt.text(start_time, 0.8, f'Lap {i+1}', fontsize=10, fontweight='bold')
        plt.axvline(x=start_time, color='red', linestyle='--', alpha=0.6)
    
    # Add horizontal line at y=0
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Add title and labels
    plt.title('QW Values Over Time with Lap Detection', fontsize=14)
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('QW Value', fontsize=12)
    plt.ylim(-1.1, 1.1)  # Set y-axis limits as shown in the image
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    
    # Add annotation about QW
    plt.figtext(0.5, 0.01, 'Note: qw is a quaternion component used to represent 3D orientation', 
                ha='center', fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.show()

# Main function
def main():
    # Path to your SQLite database
    db_path = 'b.sqlite'
    table_name = 'Orientation'
    
    # Load data from SQLite database
    time, qw = load_data_from_sqlite(db_path, table_name)
    
    # Detect laps
    threshold = 0.5  # Adjust this threshold based on your data
    lap_start_times = detect_laps(time, qw, threshold)
    
    print(f"Detected {len(lap_start_times)} laps")
    if lap_start_times:
        print("Lap start times:", lap_start_times)
        
        # Calculate lap durations
        lap_durations = []
        for i in range(len(lap_start_times)-1):
            duration = lap_start_times[i+1] - lap_start_times[i]
            lap_durations.append(duration)
        if lap_durations:
            print("Lap durations (seconds):", [round(d, 1) for d in lap_durations])
            print(f"Average lap time: {round(sum(lap_durations)/len(lap_durations), 2)} seconds")
    
    # Plot the results
    plot_laps(time, qw, lap_start_times)
    
    # Optional: Export lap timing data
    # pd.DataFrame({'lap_start_time': lap_start_times}).to_csv('lap_times.csv', index=False)

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sqlite3
from scipy.signal import find_peaks

# Load data from SQLite database
def load_data_from_sqlite(db_path='b.sqlite', table_name='Orientation'):
    """
    Load data from SQLite database
    Expects table with time and qx values
    """
    try:
        # Connect to SQLite database
        conn = sqlite3.connect(db_path)
        print(f"Connected to database: {db_path}")
        
        # First, check what columns are available
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [row[1] for row in cursor.fetchall()]
        print(f"Available columns: {columns}")
        
        # Find time column
        time_col = None
        for col in columns:
            if col.lower() in ['time', 'timestamp', 't']:
                time_col = col
                break
        
        # If no specific time column found, use first column or look for one with 'time' in it
        if not time_col:
            for col in columns:
                if 'time' in col.lower():
                    time_col = col
                    break
            if not time_col:
                time_col = columns[0]
        
        # Find qx column
        qx_col = None
        for col in columns:
            if col.lower() in ['qx', 'q1', 'q_x']:
                qx_col = col
                break
        
        # If no specific qx column found, look for one with 'qx' in it
        if not qx_col:
            for col in columns:
                if 'qx' in col.lower():
                    qx_col = col
                    break
        
        if not qx_col:
            raise ValueError(f"Could not find QX column in {columns}")
        
        print(f"Using columns: time={time_col}, qx={qx_col}")
        
        # Query data
        query = f"SELECT {time_col}, {qx_col} FROM {table_name}"
        df = pd.read_sql_query(query, conn)
        
        # Close connection
        conn.close()
        
        # Extract arrays
        time = df[time_col].values
        qx = df[qx_col].values
        
        print(f"Loaded {len(time)} data points from SQLite database")
        return time, qx
        
    except Exception as e:
        print(f"Error loading data from SQLite: {e}")
        # Generate sample data if database loading fails
        print("Generating sample data instead")
        return generate_sample_data()

def generate_sample_data():
    """Generate sample data similar to the image for testing"""
    time = np.arange(100, 800, 0.5)
    period = 120  # Approximately the period shown in the image
    
    # Base sine wave with phase shift (to mimic QX instead of QW)
    qx = np.sin(2 * np.pi * (time - 100) / period + np.pi/2) * 0.9
    
    # Add some noise and irregularities to make it look more like the real data
    noise = np.random.normal(0, 0.1, len(time))
    qx += noise
    
    # Clip to the range [-1, 1] as shown in the image
    qx = np.clip(qx, -1, 1)
    
    return time, qx

# Detect laps based on QX transitions
def detect_laps(time, qx, threshold=0.5):
    """
    Detect laps based on QX value transitions
    Using threshold crossing from negative to positive values
    """
    lap_starts = []
    for i in range(1, len(qx)):
        if qx[i-1] < threshold and qx[i] >= threshold:
            lap_starts.append(i)
    
    # Filter out laps that are too close together (likely noise)
    if len(lap_starts) > 1:
        filtered_starts = [lap_starts[0]]
        min_gap = 30  # Minimum points between lap starts
        
        for i in range(1, len(lap_starts)):
            if lap_starts[i] - filtered_starts[-1] > min_gap:
                filtered_starts.append(lap_starts[i])
        
        lap_starts = filtered_starts
    
    # Convert indices to time values
    lap_start_times = [time[i] for i in lap_starts]
    return lap_start_times

# Plot the data with lap sections highlighted
def plot_laps(time, qx, lap_start_times):
    plt.figure(figsize=(14, 7))
    
    # Plot raw QX values
    plt.plot(time, qx, 'b-', linewidth=1, label='Raw qx values')
    
    # Compute and plot a moving average to smooth the data
    window = 51
    if len(qx) > window:
        qx_smoothed = np.convolve(qx, np.ones(window)/window, mode='valid')
        time_smoothed = time[window//2:-(window//2)]
        plt.plot(time_smoothed, qx_smoothed, 'k-', linewidth=1.5, label=f'{window}-point rolling average')
    
    # Highlight lap sections
    colors = ['#ffcccc', '#ccffcc', '#ccccff', '#ffffcc', '#ffccff', '#ccffff']
    for i in range(len(lap_start_times)):
        start_time = lap_start_times[i]
        if i < len(lap_start_times) - 1:
            end_time = lap_start_times[i + 1]
        else:
            end_time = time[-1]  # Use the last time point for the final lap
            
        color = colors[i % len(colors)]
        plt.axvspan(start_time, end_time, alpha=0.2, color=color, 
                    label=f'Lap {i+1}' if i==0 else "")
    
    # Add lap number annotations
    for i, start_time in enumerate(lap_start_times):
        plt.text(start_time, 0.8, f'Lap {i+1}', fontsize=10, fontweight='bold')
        plt.axvline(x=start_time, color='red', linestyle='--', alpha=0.6)
    
    # Add horizontal line at y=0
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Add title and labels
    plt.title('QX Values Over Time with Lap Detection', fontsize=14)
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('QX Value', fontsize=12)
    plt.ylim(-1.1, 1.1)  # Set y-axis limits as shown in the image
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    
    # Add annotation about QX
    plt.figtext(0.5, 0.01, 'Note: qx is a quaternion component used to represent 3D orientation', 
                ha='center', fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.show()

# Main function
def main():
    # Path to your SQLite database
    db_path = 'b.sqlite'
    table_name = 'Orientation'
    
    # Load data from SQLite database
    time, qx = load_data_from_sqlite(db_path, table_name)
    
    # Detect laps
    threshold = 0.5  # Adjust this threshold based on your data
    lap_start_times = detect_laps(time, qx, threshold)
    
    print(f"Detected {len(lap_start_times)} laps")
    if lap_start_times:
        print("Lap start times:", lap_start_times)
        
        # Calculate lap durations
        lap_durations = []
        for i in range(len(lap_start_times)-1):
            duration = lap_start_times[i+1] - lap_start_times[i]
            lap_durations.append(duration)
        if lap_durations:
            print("Lap durations (seconds):", [round(d, 1) for d in lap_durations])
            print(f"Average lap time: {round(sum(lap_durations)/len(lap_durations), 2)} seconds")
    
    # Plot the results
    plot_laps(time, qx, lap_start_times)
    
    # Optional: Export lap timing data
    # pd.DataFrame({'lap_start_time': lap_start_times}).to_csv('lap_times.csv', index=False)

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sqlite3
from scipy.signal import find_peaks, savgol_filter

# Load data from SQLite database
def load_data_from_sqlite(db_path='b.sqlite', table_name='Orientation'):
    """
    Load data from SQLite database
    Expects table with time and qx values
    """
    try:
        # Connect to SQLite database
        conn = sqlite3.connect(db_path)
        print(f"Connected to database: {db_path}")
        
        # First, check what columns are available
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [row[1] for row in cursor.fetchall()]
        print(f"Available columns: {columns}")
        
        # Find time column
        time_col = None
        for col in columns:
            if col.lower() in ['time', 'timestamp', 't']:
                time_col = col
                break
        
        # If no specific time column found, use first column or look for one with 'time' in it
        if not time_col:
            for col in columns:
                if 'time' in col.lower():
                    time_col = col
                    break
            if not time_col:
                time_col = columns[0]
        
        # Find qx column
        qx_col = None
        for col in columns:
            if col.lower() in ['qx', 'q1', 'q_x']:
                qx_col = col
                break
        
        # If no specific qx column found, look for one with 'qx' in it
        if not qx_col:
            for col in columns:
                if 'qx' in col.lower():
                    qx_col = col
                    break
        
        if not qx_col:
            raise ValueError(f"Could not find QX column in {columns}")
        
        print(f"Using columns: time={time_col}, qx={qx_col}")
        
        # Query data
        query = f"SELECT {time_col}, {qx_col} FROM {table_name}"
        df = pd.read_sql_query(query, conn)
        
        # Close connection
        conn.close()
        
        # Extract arrays
        time = df[time_col].values
        qx = df[qx_col].values
        
        print(f"Loaded {len(time)} data points from SQLite database")
        return time, qx
        
    except Exception as e:
        print(f"Error loading data from SQLite: {e}")
        # Generate sample data if database loading fails
        print("Generating sample data instead")
        return generate_sample_data()

def generate_sample_data():
    """Generate sample data similar to the image for testing"""
    time = np.arange(100, 800, 0.5)
    period = 120  # Approximately the period shown in the image
    
    # Base sine wave with phase shift (to mimic QX instead of QW)
    qx = np.sin(2 * np.pi * (time - 100) / period + np.pi/2) * 0.9
    
    # Add some noise and irregularities to make it look more like the real data
    noise = np.random.normal(0, 0.1, len(time))
    qx += noise
    
    # Clip to the range [-1, 1] as shown in the image
    qx = np.clip(qx, -1, 1)
    
    return time, qx

# Improved lap detection based on complete cycles
def detect_laps_improved(time, qx, prominence=0.5, min_lap_time=10):
    """
    Detect laps based on full QX cycles by finding peaks and troughs
    
    Parameters:
    time (array): Time values
    qx (array): QX quaternion component values
    prominence (float): Minimum prominence for peak detection
    min_lap_time (float): Minimum time between laps in seconds
    
    Returns:
    list: List of lap start times
    """
    # Smooth the data to reduce noise
    if len(qx) > 51:
        try:
            # Try Savitzky-Golay filter first for better peak preservation
            qx_smooth = savgol_filter(qx, 51, 3)
        except:
            # Fall back to moving average if Savitzky-Golay fails
            qx_smooth = np.convolve(qx, np.ones(51)/51, mode='same')
    else:
        qx_smooth = qx
    
    # Find peaks (high points) in the data
    peaks, _ = find_peaks(qx_smooth, prominence=prominence, distance=int(min_lap_time/(time[1]-time[0])))
    
    # Find troughs (low points) in the data
    troughs, _ = find_peaks(-qx_smooth, prominence=prominence, distance=int(min_lap_time/(time[1]-time[0])))
    
    # Merge and sort all critical points
    all_points = np.sort(np.concatenate([peaks, troughs]))
    
    # Look for patterns that suggest lap boundaries
    # A lap typically has a distinctive pattern (e.g., high to low transition or specific slope)
    lap_starts = []
    
    # Based on the image, the lap start seems to be when qx has a positive peak value
    # followed by a sharp decline
    for peak_idx in peaks:
        # Check if this is a major peak (near maximum value)
        if qx_smooth[peak_idx] > 0.7:  # Threshold based on image
            # Add the peak as a lap start
            lap_starts.append(peak_idx)
    
    # If too few laps detected, try alternate approach
    if len(lap_starts) < 5:  # Expecting around 10 laps based on the description
        print("Few laps detected with peak method, trying zero-crossing approach...")
        lap_starts = []
        
        # Look for transitions from negative to positive with steep positive slope
        # This appears to match the lap boundaries in the image
        for i in range(1, len(qx_smooth)):
            if qx_smooth[i-1] < 0 and qx_smooth[i] >= 0 and (i+5 < len(qx_smooth) and qx_smooth[i+5] > 0.5):
                lap_starts.append(i)
    
    # Filter out laps that are too close together
    if len(lap_starts) > 1:
        filtered_starts = [lap_starts[0]]
        min_samples = int(min_lap_time / (time[1] - time[0]))
        
        for i in range(1, len(lap_starts)):
            if lap_starts[i] - filtered_starts[-1] > min_samples:
                filtered_starts.append(lap_starts[i])
        
        lap_starts = filtered_starts
    
    # Convert indices to time values
    lap_start_times = [time[i] for i in lap_starts]
    
    print(f"Detected {len(lap_start_times)} laps")
    return lap_start_times

# Plot the data with lap sections highlighted
def plot_laps(time, qx, lap_start_times):
    plt.figure(figsize=(14, 7))
    
    # Plot raw QX values
    plt.plot(time, qx, 'b-', linewidth=1, label='Raw qx values')
    
    # Compute and plot a moving average to smooth the data
    window = 51
    if len(qx) > window:
        qx_smoothed = np.convolve(qx, np.ones(window)/window, mode='valid')
        time_smoothed = time[window//2:-(window//2)]
        plt.plot(time_smoothed, qx_smoothed, 'k-', linewidth=1.5, label=f'{window}-point rolling average')
    
    # Highlight lap sections
    colors = ['#ffcccc', '#ccffcc', '#ccccff', '#ffffcc', '#ffccff', '#ccffff', '#ffdddd', '#ddffdd', '#ddddff', '#ffffdd']
    for i in range(len(lap_start_times)):
        start_time = lap_start_times[i]
        if i < len(lap_start_times) - 1:
            end_time = lap_start_times[i + 1]
        else:
            end_time = time[-1]  # Use the last time point for the final lap
            
        color = colors[i % len(colors)]
        plt.axvspan(start_time, end_time, alpha=0.2, color=color, 
                    label=f'Lap {i+1}' if i==0 else "")
    
    # Add lap number annotations
    for i, start_time in enumerate(lap_start_times):
        y_pos = 0.8 if i % 2 == 0 else 0.7  # Alternate positions to avoid overlap
        plt.text(start_time, y_pos, f'Lap {i+1}', fontsize=9, fontweight='bold')
        plt.axvline(x=start_time, color='red', linestyle='--', alpha=0.6)
    
    # Add horizontal line at y=0
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Add title and labels
    plt.title('QX Values Over Time with Lap Detection', fontsize=14)
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('QX Value', fontsize=12)
    plt.ylim(-1.1, 1.1)  # Set y-axis limits as shown in the image
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    
    # Add annotation about QX
    plt.figtext(0.5, 0.01, 'Note: qx is a quaternion component used to represent 3D orientation', 
                ha='center', fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.show()

# Main function
def main():
    # Path to your SQLite database
    db_path = 'b.sqlite'
    table_name = 'Orientation'
    
    # Load data from SQLite database
    time, qx = load_data_from_sqlite(db_path, table_name)
    
    # Use improved lap detection
    lap_start_times = detect_laps_improved(time, qx, prominence=0.4, min_lap_time=20)
    
    if lap_start_times:
        print("Lap start times:", lap_start_times)
        
        # Calculate lap durations
        lap_durations = []
        for i in range(len(lap_start_times)-1):
            duration = lap_start_times[i+1] - lap_start_times[i]
            lap_durations.append(duration)
        if lap_durations:
            print("Lap durations (seconds):", [round(d, 1) for d in lap_durations])
            print(f"Average lap time: {round(sum(lap_durations)/len(lap_durations), 2)} seconds")
    
    # Plot the results
    plot_laps(time, qx, lap_start_times)
    
    # Optional: Export lap timing data
    # pd.DataFrame({'lap_start_time': lap_start_times}).to_csv('lap_times.csv', index=False)

if __name__ == "__main__":
    main()

In [ ]:
import sqlite3
import pandas as pd

def get_laps_from_db(db_path, yaw_threshold_degrees=350):
    """
    Identifies laps from karting telemetry data based on yaw rotation.

    Args:
        db_path (str): The path to the SQLite database file.
        yaw_threshold_degrees (int): The threshold in degrees for a complete lap.

    Returns:
        pd.DataFrame: A DataFrame with lap start and end times.
    """
    try:
        # Connect to the SQLite database and read the data
        with sqlite3.connect(db_path) as conn:
            df = pd.read_sql_query("SELECT seconds_elapsed, yaw FROM Orientation ORDER BY seconds_elapsed", conn)
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None

    if df.empty:
        print("No data found in the Orientation table.")
        return None

    # Convert yaw to a continuous value to track full rotations
    # This assumes yaw is in degrees and resets from 360 to 0 or vice versa
    df['yaw_diff'] = df['yaw'].diff().fillna(0)
    df['continuous_yaw'] = df['yaw_diff'].apply(
        lambda x: x if abs(x) < yaw_threshold_degrees else x - 360 * (x / abs(x))
    ).cumsum()

    laps = []
    lap_start_time = df.iloc[0]['seconds_elapsed']
    lap_start_continuous_yaw = df.iloc[0]['continuous_yaw']

    # Iterate through the data to find lap completion
    for i in range(1, len(df)):
        current_time = df.iloc[i]['seconds_elapsed']
        current_yaw = df.iloc[i]['continuous_yaw']
        
        # A lap is completed when the continuous yaw changes by roughly 360 degrees
        # The threshold is set to a degree slightly less than 360 to account for noise
        if abs(current_yaw - lap_start_continuous_yaw) >= yaw_threshold_degrees:
            laps.append({
                'lap_number': len(laps) + 1,
                'start_time': lap_start_time,
                'end_time': current_time,
                'lap_duration': current_time - lap_start_time
            })
            
            # Set the start of the next lap
            lap_start_time = current_time
            lap_start_continuous_yaw = current_ya

    return pd.DataFrame(laps)

if __name__ == '__main__':
    database_file = 'b.sqlite'
    lap_data = get_laps_from_db(database_file)
    
    if lap_data is not None and not lap_data.empty:
        print("Laps identified:")
        print(lap_data)
    else:
        print("Could not identify any laps.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sqlite3
from scipy.signal import find_peaks, savgol_filter

# Load data from SQLite database
def load_data_from_sqlite(db_path='b.sqlite', table_name='Orientation'):
    """
    Load data from SQLite database
    Expects table with time and qx values
    """
    try:
        # Connect to SQLite database
        conn = sqlite3.connect(db_path)
        print(f"Connected to database: {db_path}")
        
        # First, check what columns are available
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [row[1] for row in cursor.fetchall()]
        print(f"Available columns: {columns}")
        
        # Find time column
        time_col = None
        for col in columns:
            if col.lower() in ['time', 'timestamp', 't']:
                time_col = col
                break
        
        # If no specific time column found, use first column or look for one with 'time' in it
        if not time_col:
            for col in columns:
                if 'time' in col.lower():
                    time_col = col
                    break
            if not time_col:
                time_col = columns[0]
        
        # Find qx column
        qx_col = None
        for col in columns:
            if col.lower() in ['qx', 'q1', 'q_x']:
                qx_col = col
                break
        
        # If no specific qx column found, look for one with 'qx' in it
        if not qx_col:
            for col in columns:
                if 'qx' in col.lower():
                    qx_col = col
                    break
        
        if not qx_col:
            raise ValueError(f"Could not find QX column in {columns}")
        
        print(f"Using columns: time={time_col}, qx={qx_col}")
        
        # Query data
        query = f"SELECT {time_col}, {qx_col} FROM {table_name}"
        df = pd.read_sql_query(query, conn)
        
        # Close connection
        conn.close()
        
        # Extract arrays
        time = df[time_col].values
        qx = df[qx_col].values
        
        print(f"Loaded {len(time)} data points from SQLite database")
        return time, qx
        
    except Exception as e:
        print(f"Error loading data from SQLite: {e}")
        # Generate sample data if database loading fails
        print("Generating sample data instead")
        return generate_sample_data()

def generate_sample_data():
    """Generate sample data similar to the image for testing"""
    time = np.arange(100, 800, 0.5)
    period = 120  # Approximately the period shown in the image
    
    # Base sine wave with phase shift (to mimic QX instead of QW)
    qx = np.sin(2 * np.pi * (time - 100) / period + np.pi/2) * 0.9
    
    # Add some noise and irregularities to make it look more like the real data
    noise = np.random.normal(0, 0.1, len(time))
    qx += noise
    
    # Clip to the range [-1, 1] as shown in the image
    qx = np.clip(qx, -1, 1)
    
    return time, qx

# Improved lap detection based on complete cycles
def detect_laps_improved(time, qx, prominence=0.5, min_lap_time=10):
    """
    Detect laps based on full QX cycles by finding peaks and troughs
    
    Parameters:
    time (array): Time values
    qx (array): QX quaternion component values
    prominence (float): Minimum prominence for peak detection
    min_lap_time (float): Minimum time between laps in seconds
    
    Returns:
    list: List of lap start times
    """
    # Check if we have enough data points
    if len(time) < 2:
        print("Not enough data points for lap detection")
        return []
    
    # Calculate time interval (to help with finding appropriate window size)
    time_interval = time[1] - time[0]
    print(f"Time interval between points: {time_interval} seconds")
    
    # Ensure min_lap_time is appropriate
    samples_per_lap = max(1, int(min_lap_time / time_interval))
    print(f"Minimum lap samples: {samples_per_lap}")
    
    # Smooth the data to reduce noise
    window_size = min(51, len(qx) // 10 * 2 + 1)  # Must be odd and less than data length
    window_size = max(5, window_size)  # At least 5 points
    
    try:
        # Try Savitzky-Golay filter first for better peak preservation
        qx_smooth = savgol_filter(qx, window_size, 3)
    except:
        # Fall back to moving average if Savitzky-Golay fails
        print("Savitzky-Golay filter failed, using moving average")
        window = np.ones(window_size) / window_size
        qx_smooth = np.convolve(qx, window, mode='same')
    
    # Find peaks (high points) in the data
    # Ensure distance parameter is at least 1
    min_peak_distance = max(1, int(min_lap_time / time_interval))
    
    try:
        peaks, _ = find_peaks(qx_smooth, prominence=prominence, distance=min_peak_distance)
        # Find troughs (low points) in the data
        troughs, _ = find_peaks(-qx_smooth, prominence=prominence, distance=min_peak_distance)
    except Exception as e:
        print(f"Error in peak detection: {e}")
        print("Trying alternative approach...")
        peaks = []
        troughs = []
        # Manual peak detection as fallback
        for i in range(1, len(qx_smooth)-1):
            if qx_smooth[i] > qx_smooth[i-1] and qx_smooth[i] > qx_smooth[i+1] and qx_smooth[i] > 0.7:
                peaks = np.append(peaks, i)
    
    # If peaks were found, use them to identify laps
    lap_starts = []
    if len(peaks) > 0:
        print(f"Found {len(peaks)} peaks in the data")
        # Based on the image, the lap start seems to be when qx has a positive peak
        for peak_idx in peaks:
            if qx_smooth[int(peak_idx)] > 0.7:  # High positive peak
                lap_starts.append(int(peak_idx))
    else:
        print("No peaks found, trying zero-crossing approach")
        # Alternative approach: look for zero-crossings with steep positive slopes
        for i in range(1, len(qx_smooth)):
            # Check for crossing from negative to positive with steep slope
            if qx_smooth[i-1] < 0 and qx_smooth[i] >= 0:
                # Check if it's followed by a continued rise
                if i+5 < len(qx_smooth) and qx_smooth[i+5] > 0.5:
                    lap_starts.append(i)
    
    # Filter out laps that are too close together
    if len(lap_starts) > 1:
        filtered_starts = [lap_starts[0]]
        for i in range(1, len(lap_starts)):
            if lap_starts[i] - filtered_starts[-1] > min_peak_distance:
                filtered_starts.append(lap_starts[i])
        
        lap_starts = filtered_starts
    
    # Convert indices to time values
    lap_start_times = [time[i] for i in lap_starts]
    
    print(f"Detected {len(lap_start_times)} laps")
    return lap_start_times

# Plot the data with lap sections highlighted
def plot_laps(time, qx, lap_start_times):
    plt.figure(figsize=(14, 7))
    
    # Plot raw QX values
    plt.plot(time, qx, 'b-', linewidth=1, label='Raw qx values')
    
    # Compute and plot a moving average to smooth the data
    window = min(51, len(qx) // 10)
    window = max(3, window)  # At least 3 points
    if len(qx) > window:
        qx_smoothed = np.convolve(qx, np.ones(window)/window, mode='same')
        plt.plot(time, qx_smoothed, 'k-', linewidth=1.5, label=f'{window}-point rolling average')
    
    # Highlight lap sections
    colors = ['#ffcccc', '#ccffcc', '#ccccff', '#ffffcc', '#ffccff', '#ccffff', '#ffdddd', '#ddffdd', '#ddddff', '#ffffdd']
    for i in range(len(lap_start_times)):
        start_time = lap_start_times[i]
        if i < len(lap_start_times) - 1:
            end_time = lap_start_times[i + 1]
        else:
            end_time = time[-1]  # Use the last time point for the final lap
            
        color = colors[i % len(colors)]
        plt.axvspan(start_time, end_time, alpha=0.2, color=color, 
                    label=f'Lap {i+1}' if i==0 else "")
    
    # Add lap number annotations
    for i, start_time in enumerate(lap_start_times):
        y_pos = 0.8 if i % 2 == 0 else 0.7  # Alternate positions to avoid overlap
        plt.text(start_time, y_pos, f'Lap {i+1}', fontsize=9, fontweight='bold')
        plt.axvline(x=start_time, color='red', linestyle='--', alpha=0.6)
    
    # Add horizontal line at y=0
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Add title and labels
    plt.title('QX Values Over Time with Lap Detection', fontsize=14)
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('QX Value', fontsize=12)
    plt.ylim(-1.1, 1.1)  # Set y-axis limits as shown in the image
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    
    # Add annotation about QX
    plt.figtext(0.5, 0.01, 'Note: qx is a quaternion component used to represent 3D orientation', 
                ha='center', fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.show()

# Main function
def main():
    # Path to your SQLite database
    db_path = 'b.sqlite'
    table_name = 'Orientation'
    
    # Load data from SQLite database
    time, qx = load_data_from_sqlite(db_path, table_name)
    
    # Use improved lap detection with safer parameters
    # Adjust these parameters if needed
    lap_start_times = detect_laps_improved(time, qx, prominence=0.4, min_lap_time=5)
    
    if lap_start_times:
        print("Lap start times:", lap_start_times)
        
        # Calculate lap durations
        lap_durations = []
        for i in range(len(lap_start_times)-1):
            duration = lap_start_times[i+1] - lap_start_times[i]
            lap_durations.append(duration)
        if lap_durations:
            print("Lap durations (seconds):", [round(d, 1) for d in lap_durations])
            print(f"Average lap time: {round(sum(lap_durations)/len(lap_durations), 2)} seconds")
    
    # Plot the results
    plot_laps(time, qx, lap_start_times)
    
    # Optional: Export lap timing data
    # pd.DataFrame({'lap_start_time': lap_start_times}).to_csv('lap_times.csv', index=False)

if __name__ == "__main__":
    main()

In [ ]:
# load Location, Orientation, Compass, Accelerometer, Gravity, Gyroscope, Magnetometer, WristMotion
# fuse the data, interpolating the positions
SECS_ELAPSED_START = 88
SECS_ELAPSED_END = 1046